# LLM-Assisted Analog Amplifier Sizing

**An LLM-guided analog amplifier sizing flow using gm/ID methodology, ngspice SPICE simulation, and iterative root-cause diagnosis on the SKY130 open-source PDK.**

---

## Team Members

| Name | Affiliation | Email | IEEE Member | SSCS Member |
|------|-------------|-------|-------------|-------------|
| Jiyuan Duan    | Rice University              | jd200@rice.edu           | Yes | Yes |
| Shikai Wang    | George Washington University | shikai.wang@gwu.edu      | No  | No  |
| Gerald Topalli | Rice University              | gerald.topalli@rice.edu  | Yes | Yes |
| Houbo He       | Rice University              | houbo.he@rice.edu        | Yes | Yes |
| Lei Xia        | Rice University              | lx27@rice.edu            | Yes | Yes |
| Weidong Cao    | George Washington University | weidong.cao@gwu.edu      | Yes | Yes |
| Taiyun Chi     | Rice University              | tc57@rice.edu            | Yes | Yes |

## References

- B. Murmann, "gm/ID-Based Design Methodology" (Stanford)
- SkyWater SKY130 Open-Source PDK: https://github.com/google/skywater-pdk
- ngspice Open-Source SPICE Simulator: https://ngspice.sourceforge.io/
- Claude Code: https://claude.ai/claude-code

## Tool Versions

| Tool | Version | Purpose |
|------|---------|---------|  
| ngspice | 46 | SPICE simulation |
| SKY130 PDK | Open-source | 130nm process (bundled in repo) |
| Python | 3.11 | Agent logic, gm/ID LUT queries |
| Claude Code | Latest | LLM-driven interactive design agent (VS Code extension) |
| FastAPI | 0.118+ | CircuitCollector simulation server |
| Conda/Miniforge | Latest | Environment management |

## License

Apache 2.0

---

## Overview

This notebook demonstrates **AnalogAgent**, an LLM-guided system for automated analog amplifier sizing. The system combines:

1. **gm/ID methodology** with pre-computed look-up tables (LUTs) for SKY130 devices
2. **SPICE verification** via ngspice through an automated testbench generation server (CircuitCollector)
3. **Iterative root-cause diagnosis** where the LLM agent identifies spec failures and adjusts device parameters
4. **CMA-ES numerical optimization** for post-sizing fine-tuning

### Architecture

```
User (specs + netlist)
        |
        v
+------------------+     HTTP (port 8001)     +--------------------+
|   AnalogAgent    | <---------------------> | CircuitCollector    |
|  (Claude Code    |                          |                    |
|   in VS Code)    |    /simulate/            | - Testbench gen    |
| - LLM reasoning  |    /register_circuit/    | - ngspice runner   |
| - gm/ID LUT      |    /health               | - Result parser    |
| - Skill stack     |                          | - SKY130 PDK       |
| - CMA-ES optimizer|                          |                    |
+------------------+                          +--------------------+
```

### Design Flow

```
Specs (VDD, CL, Gain, GBW, PM, ...)
  --> [1] Spec understanding & validation
  --> [2] Circuit topology identification (from user netlist)
  --> [3] Topology registration with CircuitCollector
  --> [4] Initial gm/ID-based analytical sizing
  --> [5] LUT-driven parameter conversion (gm/ID targets -> W, L, M)
  --> [6] SPICE simulation via ngspice
  --> [7] Spec verification & root-cause diagnosis
  --> [8] Iterate (steps 4-7) until specs met or max iterations
  --> [9] Optional CMA-ES numerical optimization
  --> [10] Final design review with PVT analysis
```

### Supported Topologies

| Topology | Variants |
|----------|----------|
| 5-Transistor OTA | Single / Cascode / Wide-swing cascode load |
| Two-Stage Miller (TSM) | Single / Cascode / Wide-swing cascode load |

---

### How This Notebook Works

This notebook **sets up the complete environment** (ngspice, conda, repos, server) and then guides you to open **Claude Code in VS Code** for the interactive sizing flow. The actual design happens inside Claude Code, where the LLM agent reasons through the analog design process step by step.

**Sections:**
1. Environment setup (system deps, ngspice, conda, repo)
2. Start and verify CircuitCollector simulation server
3. Install VS Code + Claude Code and run the interactive sizing agent

---

## Section 1: Environment Setup

This section installs all prerequisites on a fresh Ubuntu/Debian system:
1. System build dependencies
2. ngspice 46 (built from source)
3. Miniforge (conda)
4. Clone the project repository
5. Create two separate conda environments

> **Note:** If you already have any of these installed, you can skip the corresponding cells.

### 1.1 Install System Build Dependencies

In [ ]:
%%bash
sudo apt-get update -qq
sudo apt-get install -y -qq \
    build-essential autoconf automake libtool bison flex \
    libx11-dev libxaw7-dev libreadline-dev libfftw3-dev \
    wget curl git \
    2>&1 | tail -3
echo "System dependencies installed."

### 1.2 Build and Install ngspice 46

ngspice is the open-source SPICE simulator used for all circuit simulations. We build version 46 from source and install it to `$HOME/ngspice46/`.

In [ ]:
%%bash
set -e

NGSPICE_VERSION=46
INSTALL_DIR="$HOME/ngspice${NGSPICE_VERSION}"

echo "=== Downloading ngspice-${NGSPICE_VERSION} source ==="
cd /tmp
wget -q "https://sourceforge.net/projects/ngspice/files/ng-spice-rework/${NGSPICE_VERSION}/ngspice-${NGSPICE_VERSION}.tar.gz/download" \
     -O ngspice-${NGSPICE_VERSION}.tar.gz
tar xzf ngspice-${NGSPICE_VERSION}.tar.gz
cd ngspice-${NGSPICE_VERSION}

echo "=== Configuring ==="
mkdir -p release && cd release
../configure \
    --prefix="${INSTALL_DIR}" \
    --with-x \
    --with-readline=yes \
    --with-fftw3=yes \
    --enable-xspice \
    --enable-cider \
    --enable-openmp \
    --enable-klu \
    > /dev/null 2>&1

echo "=== Compiling (this may take a few minutes) ==="
make -j$(nproc) > /dev/null 2>&1

echo "=== Installing to ${INSTALL_DIR} ==="
make install > /dev/null 2>&1

# Verify
${INSTALL_DIR}/bin/ngspice --version 2>&1 | head -5
echo ""
echo "ngspice installed at ${INSTALL_DIR}/bin/ngspice"

In [ ]:
import os, shutil

NGSPICE_BIN_DIR = os.path.expanduser("~/ngspice46/bin")

# Add ngspice to PATH for this notebook session and all child processes
os.environ["PATH"] = NGSPICE_BIN_DIR + ":" + os.environ["PATH"]

ngspice_path = shutil.which("ngspice")
assert ngspice_path is not None, f"ngspice not found! Expected at {NGSPICE_BIN_DIR}/ngspice"
print(f"ngspice ready: {ngspice_path}")

In [ ]:
import os

# Write ~/.spiceinit for SKY130 PDK
# Only num_threads is active; all compatibility/perf options are commented out
spiceinit_path = os.path.expanduser("~/.spiceinit")
spiceinit_content = """\
* .spiceinit for use with Skywater PDK and ngspice
* set ngbehavior=hsa   ; HSA mode disabled
* set skywaterpdk      ; skip some checks for faster lib loading
* set ng_nomodcheck    ; don't check the model parameters
set num_threads=8      ; CPU processor cores available
* option noinit        ; don't print operating point data
* option klu           ; select KLU as matrix solver
"""

with open(spiceinit_path, "w") as f:
    f.write(spiceinit_content)

print(f"Written {spiceinit_path}")
print(open(spiceinit_path).read())

### 1.3 Install Miniforge (Conda)

Two separate conda environments are needed — one for the simulation server, one for the agent.

In [ ]:
%%bash
set -e

# Check if conda is available (command -v, or check common install paths)
if command -v conda &> /dev/null; then
    echo "Conda already installed: $(conda --version)"
    exit 0
fi

for DIR in "$HOME/miniforge3" "$HOME/anaconda3" "$HOME/miniconda3"; do
    if [ -x "${DIR}/bin/conda" ]; then
        echo "Conda already installed at ${DIR}"
        ${DIR}/bin/conda --version
        exit 0
    fi
done

echo "=== Installing Miniforge ==="
wget -q "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh" \
     -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p "$HOME/miniforge3"
rm /tmp/miniforge.sh

echo "=== Initializing conda ==="
"$HOME/miniforge3/bin/conda" init bash
echo "Miniforge installed. You may need to restart your shell for conda to be available."

In [ ]:
import os, shutil, subprocess

# Find conda executable
conda_path = shutil.which("conda")
if not conda_path:
    # Check common locations
    for candidate in [
        os.path.expanduser("~/miniforge3/bin/conda"),
        os.path.expanduser("~/anaconda3/bin/conda"),
        os.path.expanduser("~/miniconda3/bin/conda"),
        "/usr/bin/conda",
    ]:
        if os.path.isfile(candidate):
            conda_path = candidate
            os.environ["PATH"] = os.path.dirname(candidate) + ":" + os.environ["PATH"]
            break

assert conda_path, "conda not found! Please install Miniforge and restart the notebook."

# Store for later use
CONDA = conda_path
print(f"conda ready: {CONDA}")
r = subprocess.run([CONDA, "--version"], capture_output=True, text=True)
print(r.stdout.strip())

### 1.4 Clone the Project Repository

The repository contains **AnalogAgent** and **CircuitCollector** as parallel subfolders, along with the SKY130 PDK and pre-generated gm/ID look-up tables.

In [ ]:
%%bash
set -e

REPO_URL="https://github.com/jiyuanduan001-oss/LLM-Assisted-Analog-Amplifier-Sizing.git"
REPO_DIR="$HOME/LLM-Assisted-Analog-Amplifier-Sizing"

if [ -d "${REPO_DIR}" ]; then
    echo "Repository already cloned at ${REPO_DIR}"
else
    echo "=== Cloning repository ==="
    git clone "${REPO_URL}" "${REPO_DIR}"
fi

echo ""
echo "Repository structure:"
echo "  ${REPO_DIR}/"
echo "  ├── AnalogAgent/       (LLM agent + gm/ID LUT + skill stack)"
echo "  └── CircuitCollector/  (simulation server + ngspice runner + SKY130 PDK)"

In [ ]:
import os

REPO_DIR = os.path.expanduser("~/LLM-Assisted-Analog-Amplifier-Sizing")
ANALOG_AGENT_DIR = os.path.join(REPO_DIR, "AnalogAgent")
CIRCUIT_COLLECTOR_DIR = os.path.join(REPO_DIR, "CircuitCollector")

assert os.path.isdir(ANALOG_AGENT_DIR), f"AnalogAgent not found at {ANALOG_AGENT_DIR}"
assert os.path.isdir(CIRCUIT_COLLECTOR_DIR), f"CircuitCollector not found at {CIRCUIT_COLLECTOR_DIR}"

print(f"AnalogAgent:      {ANALOG_AGENT_DIR}")
print(f"CircuitCollector: {CIRCUIT_COLLECTOR_DIR}")

### 1.5 Create Conda Environments

Two **separate** conda environments are required:

| Environment | Purpose | Key Packages |
|-------------|---------|------|
| `CircuitCollector` | Runs the FastAPI simulation server, calls ngspice | FastAPI, uvicorn, jinja2, scipy |
| `Agent` | Runs the LLM agent, gm/ID LUT queries | anthropic, httpx, pandas, numpy |

> **Important:** Using a single environment will NOT work. The CircuitCollector server must run in its own environment.

In [ ]:
import subprocess, os

REPO_DIR = os.path.expanduser("~/LLM-Assisted-Analog-Amplifier-Sizing")

# Required pip packages for CircuitCollector
CC_PIP_PACKAGES = [
    "fastapi==0.118.0", "uvicorn==0.35.0", "httpx==0.28.1",
    "jinja2==3.1.6", "toml==0.10.2", "numpy==2.2.6", "pandas==2.2.3",
    "scipy==1.16.2", "redis==7.1.0", "python-dotenv==1.1.1",
    "pydantic==2.11.7", "pydantic-settings==2.10.1", "sqlalchemy==2.0.41",
    "requests==2.32.5", "matplotlib==3.10.7",
]

# Check if CircuitCollector env already exists
r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
if "CircuitCollector" in r.stdout:
    print("CircuitCollector environment already exists.")
else:
    # Try creating from environment.yml first
    env_file = os.path.join(REPO_DIR, "CircuitCollector", "CircuitCollector", "environment.yml")
    print("Creating CircuitCollector environment (this may take several minutes)...")
    r = subprocess.run(
        [CONDA, "env", "create", "-f", env_file],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        # Fallback: create a clean Python 3.11 environment
        print("environment.yml failed (platform mismatch). Falling back to clean env...")
        subprocess.run(
            [CONDA, "create", "-n", "CircuitCollector", "python=3.11", "-y"],
            capture_output=True, text=True
        )
    print("Base environment created.")

# Unconditionally install all required pip packages as a safety net
print("Installing required pip packages...")
r = subprocess.run(
    [CONDA, "run", "-n", "CircuitCollector",
     "pip", "install", "--quiet"] + CC_PIP_PACKAGES,
    capture_output=True, text=True
)
if r.returncode != 0:
    print(f"pip install warning:\n{r.stderr[-300:]}")
else:
    print("Pip packages installed.")

# Install CircuitCollector as editable package
print("Installing CircuitCollector package...")
r = subprocess.run(
    [CONDA, "run", "-n", "CircuitCollector",
     "pip", "install", "-e", os.path.join(REPO_DIR, "CircuitCollector"), "--quiet"],
    capture_output=True, text=True
)
print("Done." if r.returncode == 0 else f"Error: {r.stderr[-300:]}")

In [ ]:
import subprocess, os

REPO_DIR = os.path.expanduser("~/LLM-Assisted-Analog-Amplifier-Sizing")

# Check if Agent env already exists
r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
if "Agent" in r.stdout:
    print("Agent environment already exists.")
else:
    print("Creating Agent environment...")
    env_file = os.path.join(REPO_DIR, "AnalogAgent", "environment.yml")
    r = subprocess.run(
        [CONDA, "env", "create", "-f", env_file],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"Error:\n{r.stderr[-500:]}")
    else:
        print("Agent environment created.")

# Verify both environments
r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
for line in r.stdout.strip().split("\n"):
    if "CircuitCollector" in line or "Agent" in line:
        print(f"  {line.strip()}")

---

## Section 2: Start and Verify CircuitCollector Server

CircuitCollector is a FastAPI server that handles:
- Dynamic circuit registration (converting SPICE netlists to parameterized templates)
- Testbench generation using Jinja2 templates
- ngspice simulation execution
- Result parsing (AC, DC, noise, slew rate, output swing, operating point data)

The server runs in the **CircuitCollector** conda environment with ngspice on the PATH.

In [ ]:
import subprocess, os, time

REPO_DIR = os.path.expanduser("~/LLM-Assisted-Analog-Amplifier-Sizing")
CIRCUIT_COLLECTOR_DIR = os.path.join(REPO_DIR, "CircuitCollector")
NGSPICE_BIN_DIR = os.path.expanduser("~/ngspice46/bin")
SERVER_LOG = "/tmp/circuitcollector.log"
SERVER_PID_FILE = "/tmp/circuitcollector.pid"

# Stop any existing CircuitCollector server so we start fresh
subprocess.run(
    "pkill -f 'CircuitCollector.api.main:app' 2>/dev/null; sleep 1; true",
    shell=True,
)

# Start the CircuitCollector server fully DETACHED from this notebook kernel.
#
# Why detached:
#   Claude Code (launched later in a separate terminal / VS Code) calls
#   simulate_circuit() via HTTP. If the server is a child of this notebook,
#   closing the notebook or restarting the kernel kills the server.
#
#   nohup  -> ignore SIGHUP when the parent shell exits
#   setsid -> detach from the controlling terminal (new session)
#   < /dev/null, >log, 2>&1 -> close stdin, redirect stdout/stderr
#   &, disown -> don't block, remove from shell's job table
#
# Note: bash -c uses single quotes so $PATH is expanded by bash at run time.
# $PATH is wrapped in double quotes because PATH inherited from Jupyter may
# contain directories with spaces (e.g. "/mnt/d/Microsoft VS Code/bin" on WSL).
cmd = (
    f"nohup setsid bash -c '"
    f'export PATH="{NGSPICE_BIN_DIR}:$PATH" && '
    f"cd {CIRCUIT_COLLECTOR_DIR} && "
    f"exec {CONDA} run --no-capture-output -n CircuitCollector "
    f"uvicorn CircuitCollector.api.main:app --host 0.0.0.0 --port 8001"
    f"' > {SERVER_LOG} 2>&1 < /dev/null & "
    f"echo $! > {SERVER_PID_FILE}; disown"
)
subprocess.run(cmd, shell=True, executable="/bin/bash")
time.sleep(5)

with open(SERVER_PID_FILE) as f:
    server_pid = int(f.read().strip())

print(f"CircuitCollector server started (PID: {server_pid}, detached)")
print(f"  Log:          {SERVER_LOG}")
print(f"  PID file:     {SERVER_PID_FILE}")
print(f"  ngspice PATH: {NGSPICE_BIN_DIR}")
print("")
print("Server will keep running after this notebook closes.")
print("To stop it manually later:  kill $(cat /tmp/circuitcollector.pid)")

In [ ]:
import urllib.request, json, time

SERVER_URL = "http://localhost:8001"
max_retries = 10

for i in range(max_retries):
    try:
        r = urllib.request.urlopen(f"{SERVER_URL}/health", timeout=5)
        data = json.loads(r.read())
        print(f"CircuitCollector server is running: {data}")
        break
    except Exception:
        print(f"  Waiting... ({i+1}/{max_retries})")
        time.sleep(3)
else:
    # Server didn't come up — surface the log to help debugging
    log_path = "/tmp/circuitcollector.log"
    if os.path.exists(log_path):
        with open(log_path) as f:
            tail = "".join(f.readlines()[-30:])
        print(f"Last 30 lines of {log_path}:\n{tail}")
    raise RuntimeError(
        "CircuitCollector server failed to start. "
        "Check /tmp/circuitcollector.log for details."
    )

### Quick Simulation Test

Verify the full pipeline works: send a test parameter set to CircuitCollector and confirm we get back simulation results from ngspice.

In [ ]:
import urllib.request, json

SERVER_URL = "http://localhost:8001"

# Test with a 5T OTA parameter set
payload = json.dumps({
    "params": {
        "M1_L": 0.5,  "M1_WL_ratio": 5.0,  "M1_M": 1,
        "M2_L": 0.5,  "M2_WL_ratio": 5.0,  "M2_M": 1,
        "M3_L": 0.5,  "M3_WL_ratio": 5.0,  "M3_M": 1,
        "M4_L": 0.5,  "M4_WL_ratio": 5.0,  "M4_M": 1,
        "M5_L": 1.0,  "M5_WL_ratio": 5.0,  "M5_M": 1,
        "M6_L": 1.0,  "M6_WL_ratio": 5.0,  "M6_M": 1,
        "ibias": 5e-6,
    },
    "base_config_path": "config/skywater/opamp/5tota_single.toml",
    "spec_list": ["AC", "DC", "GBW_PM"],
}).encode()

print("Sending test simulation to CircuitCollector...")
req = urllib.request.Request(
    f"{SERVER_URL}/simulate/",
    data=payload,
    headers={"Content-Type": "application/json"},
)
r = urllib.request.urlopen(req, timeout=120)
result = json.loads(r.read())

if "specs" in result:
    print("\nSimulation successful!\n")
    specs = result["specs"]
    print("Key results:")
    for key, unit, scale in [
        ("dcgain_", "dB", 1),
        ("gain_bandwidth_product_", "MHz", 1e-6),
        ("phase_margin", "deg", 1),
        ("power", "uW", 1e6),
    ]:
        val = specs.get(key)
        if val is not None:
            print(f"  {key}: {val * scale:.2f} {unit}")
    print("\nCircuitCollector <-> ngspice pipeline is working correctly.")
else:
    print("Unexpected result:")
    print(json.dumps(result, indent=2)[:500])

---

## Section 3: Install VS Code + Claude Code and Run the Sizing Agent

The sizing flow runs interactively inside **Claude Code**, integrated as a VS Code extension. Claude Code loads the AnalogAgent skill stack and uses it to reason through the analog design process — choosing gm/ID operating points, querying the LUT, running simulations, diagnosing failures, and iterating until specs are met.

### 3.1 Install VS Code and Claude Code Extension

In [ ]:
%%bash
set -e

# Install VS Code if not present
if command -v code &> /dev/null; then
    echo "VS Code already installed: $(code --version 2>/dev/null | head -1)"
else
    echo "=== Installing VS Code ==="
    wget -q "https://go.microsoft.com/fwlink/?LinkID=760868" -O /tmp/vscode.deb
    sudo dpkg -i /tmp/vscode.deb 2>/dev/null || sudo apt-get install -f -y -qq 2>&1 | tail -3
    rm /tmp/vscode.deb
    echo "VS Code installed."
fi

# Install Claude Code extension
echo ""
echo "=== Installing Claude Code VS Code extension ==="
code --install-extension anthropic.claude-code 2>&1 || echo "(If this fails, install it manually from the VS Code Extensions marketplace)"
echo "Done."

### 3.2 Launch the Sizing Flow

After all environments are prepared, follow these steps.

> **Important:** The CircuitCollector server started in Section 2 is detached and
> keeps running in the background, so Claude Code can reach it over HTTP even
> after this notebook is closed. If you ever need to stop it:
> ```bash
> kill $(cat /tmp/circuitcollector.pid)
> ```
> If you ever need to restart it manually (e.g., after a WSL reboot):
> ```bash
> conda activate CircuitCollector
> export PATH="$HOME/ngspice46/bin:$PATH"
> cd ~/LLM-Assisted-Analog-Amplifier-Sizing/CircuitCollector
> nohup setsid uvicorn CircuitCollector.api.main:app --host 0.0.0.0 --port 8001 \
>     > /tmp/circuitcollector.log 2>&1 < /dev/null &
> echo $! > /tmp/circuitcollector.pid
> disown
> ```

**Step 1: Activate the Agent environment and open VS Code**

```bash
conda activate Agent
cd ~/LLM-Assisted-Analog-Amplifier-Sizing/AnalogAgent
code .
```

**Step 2: Open Claude Code inside VS Code**

- Press `Ctrl+Shift+P` (or `Cmd+Shift+P` on Mac) to open the Command Palette
- Type "Claude Code" and select **Claude Code: Open**
- Or click the Claude Code icon in the VS Code sidebar

**Step 3: Log in with your Anthropic account**

- When Claude Code opens for the first time, it will prompt you to sign in
- Log in with your personal Anthropic account (Claude Pro, Max, or Team plan)
- No API key is needed — authentication is handled through your account

**Step 4: Paste the sizing prompt** (see below) into the Claude Code panel

Claude Code will automatically load the AnalogAgent skill stack from `.claude/skills/` and have access to all the gm/ID LUT data and CircuitCollector simulation tools.

### Example Prompt

**Step 1: Set your specifications**

Open `spec-form-template.md` in the repo root and fill in your target specs (VDD, CL, DC gain, GBW, phase margin, slew rate, noise, etc.). This template is the single source of truth for the specs the agent will size against.

**Step 2: Choose a netlist**

Reference netlists are provided in `AnalogAgent/examples/` for every supported topology — 5T OTA (single / cascode / wide-swing cascode load) and Two-Stage Miller (single / cascode / wide-swing cascode load). For example, `examples/5tota_variants/5tota_single.sp` is the plain 5-transistor OTA.

> **Note:** You can also supply your own netlist. Just drop a `.sp` SPICE subcircuit into the working directory (or anywhere Claude Code can read it) and reference it in your prompt. The agent will parse the topology, register it with CircuitCollector, and size it with the same flow.

**Step 3: Prompt Claude Code**

Paste a prompt like the following into the Claude Code panel:

```
use analog amplifier skills to size the 5tota_single netlist, use the specs in template
```

---

> ### ⚠ IMPORTANT — Bash Command Permissions
>
> During execution, Claude will run **many bash commands** to call the ngspice simulator, read/write netlist and config files, query the gm/ID LUT, and so on.
>
> **You have two options:**
>
> 1. **Manual approval (default)** — A dialog will pop up for every command. You must click **Yes / Allow** each time for the flow to continue. Leaving the session unattended will stall the agent.
>
> 2. **Bypass mode (recommended for end-to-end runs)** — Enable **bypass permissions mode** in Claude Code so commands auto-approve. This lets the sizing flow run unattended from start to finish.

---

The agent will automatically:
1. Load specs from `spec-form-template.md` and the chosen netlist
2. Identify the topology (e.g., 5T OTA with single mirror load)
3. Register it with CircuitCollector
4. Size all devices using gm/ID methodology + SKY130 LUT data
5. Run SPICE simulation via ngspice
6. Diagnose and fix any spec failures
7. Optionally run CMA-ES numerical optimization
8. Produce a full design review with spec comparison table

---

## Appendix: AnalogAgent + CircuitCollector System Description

### 1. System Overview

**AnalogAgent + CircuitCollector** is an LLM-driven analog IC sizing platform built on the SKY130 PDK. It combines Claude AI (with extended thinking) as the decision-making engine, a gm/ID methodology for physics-informed sizing, and ngspice (via CircuitCollector's FastAPI server) for SPICE-accurate verification.

**End-to-end flow:** a user provides a SPICE netlist + performance specs → the agent assigns device roles, sizes every transistor from LUT data, simulates, diagnoses failures using topology-specific fault trees, iterates (up to 10 rounds), and optionally runs CMA-ES optimization for further improvement.

### 2. Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                         AnalogAgent                           │
│   Claude (Opus/Sonnet) + Skill Stack + Tool Executor          │
│      │                                                        │
│      ├── LUT Query (gm/ID, parasitics, vdsat, ft, vth)        │
│      ├── Sizing Converter (role → device params)              │
│      ├── CMA-ES Optimizer                                     │
│      └── HTTP ─────┐                                          │
└────────────────────┼──────────────────────────────────────────┘
                     ▼
┌──────────────────────────────────────────────────────────────┐
│                       CircuitCollector                        │
│   FastAPI :8001 → Jinja2 Testbench Generator → ngspice        │
│                → Result Parser → Redis + SQLite Cache         │
└──────────────────────────────────────────────────────────────┘
```

### 3. Currently Supported Topologies

In practice, only two topologies are production-ready today. Other topology skills exist in the repo but are not fully validated end-to-end.

| Topology | Status | Description |
|----------|--------|-------------|
| **5-Transistor OTA (5T OTA)** | Production | Differential pair + mirror load + tail + bias reference; simplest single-stage OTA |
| **Two-Stage Miller (TSM)** | Production | 1st stage (diff pair + mirror load) + 2nd stage (CS amp) with explicit Miller compensation (Rc + Cc) |
| **Folded-Cascode, Telescopic, Rail-to-Rail** | Experimental | Skill files exist, not yet production-validated |

### 4. Key Strength #1 — Sub-Block (Active Load) Abstraction

At the topology level, the framework introduces the concept of **sub-blocks** and treats each current mirror / active load as a black box exposing **five effective external quantities**. This gives the framework strong flexibility: the same topology equations work regardless of how the load is physically implemented.

**Three Supported Sub-Block Variants**

| Variant | Structure | Bias | Netlist Signature |
|---------|-----------|------|-------------------|
| **Single** | One transistor between output and rail | Internal (diode) | Default: no cascode companion |
| **Cascode (self-biased)** | Two same-type transistors stacked; both gates driven from the mirror feedback chain | Internal | Both transistor gates on the mirror diode node |
| **LV Cascode (Sooch / wide-swing)** | Two stacked same-type transistors, but main gate tied to cascode drain and cascode gate driven by an external bias port | External (`Vbias_cas_*`) | `M_main.gate == M_cas.drain`; external bias port |

**The Black-Box Interface**

Every sub-block, regardless of variant, exposes these five quantities to the surrounding topology equations:

| Quantity | Single | Cascode / LV Cascode |
|----------|--------|----------------------|
| `gds_eq` (output conductance) | `gds` | `(gds_m × gds_c) / gm_c` — boosted by intrinsic gain of cascode |
| `C_eq` (output-node cap) | `Cgd + Cdb` | `Cgd_cas + Cdb_cas` |
| `p_int` (internal pole) | none | `gm_cas / C_int` where `C_int = Cgs_cas + Cdb_main + Cgd_main` |
| `V_headroom` | `vdsat` | `vdsat_m + vdsat_c` (cascode) / `vdsat_c` (LV) |
| `Vbias_ext` | none | none / required (LV only) |

**Result:** The framework supports **3 × 3 = 9 structural variants per topology** (load × tail), while the circuit equations are written once in terms of the black-box interface.

### 5. Key Strength #2 — Detailed Equations + LUTs (Parasitics & Higher-Order Effects)

The framework provides circuit-specific symbolic equations (in the `*-equation.md` skill files) that account for parasitics and higher-order effects, driven by LUT data.

**LUT Data Provides Per-Width Parasitics**

The SKY130 gm/ID LUT stores 11 metrics indexed by (device, corner, temperature, channel length, gm/ID):

| Metric | Units | Role in Equations |
|--------|-------|-------------------|
| `id/W` | A/m | Derive device width: `W = ID / (id/W)` |
| `gm/id` | S/A | Operating-point selector |
| `gm/gds` | V/V | Compute `gds = gm / (gm/gds)` |
| `ft` | Hz | Device speed bound |
| `Cgg/W`, `Cgs/W`, `Cgd/W`, `Cdb/W` | F/m | Scale to actual device: `Cgd = (Cgd/W) × W` → feeds into pole/zero equations |
| `vdsat` | V | Saturation margin + headroom |
| `vth`, `vgs` | V | Bias point, cascode bias derivation |

**Parasitics Captured in Equations**

- **Mirror pole** (`fp2 = gm_mirror / (2π · C_mirror)`) built from `Cgs5 + Cgs6 + Cdb6 + Cgd2 + Cdb2` (with sub-block-dependent substitutions)
- **Internal cascode pole** (`p_int`) added as an `arctan(ω_c / p_int)` phase-margin penalty
- **TSM LHP zero placement** (`Rc = (1/gm7)(Cc + C1)(Cc + CTL)/Cc²`) uses sub-block-aware `C1` and `CTL`
- **Total output capacitance** (`C_out = CL + Cgd_in + Cdb_in + C_eq_LOAD`)
- **Right-half-plane zero** in TSM detected and nulled via `Rc`

This lets the agent estimate gain, GBW, and phase margin **analytically before simulation**, then reconcile differences after simulation — enabling root-cause diagnosis rather than blind iteration.

### 6. Key Strength #3 — Testbench Design Methodology

The testbench infrastructure lives in `spec_lib/opamp/simulation.j2`. Each spec has a dedicated DUT instance (`x1`–`x8`) configured for its specific measurement. All testbenches share the same parameterized Jinja2 library, so adding a new topology requires no testbench re-work.

**How Each Spec Is Simulated and Measured**

**DC Gain, GBW, Phase Margin — Instance `x1`**
- Analysis: `ac dec 10 0.1 1G` (AC sweep, 0.1 Hz to 1 GHz, 10 pts/decade)
- Config: Unity-gain feedback via pseudo-inductor `Cin` / pseudo-capacitor — breaks loop at AC but holds DC bias at `VCM_ratio × VDD`
- Measurements:
    - `dcgain = |vdb(opout) @ 0.1 Hz|`
    - `gain_bandwidth_product = freq when vdb(opout) = 0`
    - `phase_margin = vp(opout) when vdb(opout) = 0`
- **Zero-plateau correction:** If a high-frequency zero inflates the 0-dB crossing, a check at `f = GBW/10` measures the actual gain vs. the expected +20 dB (ideal −20 dB/dec slope). Deficit `gain_peaking_dB` is reported, and `true_GBW = GBW / 10^(peaking_dB/20)` gives the corrected value.

**CMRR — Instance `x2`**
- Analysis: Same AC sweep
- Config: Two matched AC sources drive both `vinp` and `vinn` with the same 1V common-mode signal
- Measurement: `cmrr = −vdb(cm3) @ 0.1 Hz` (negated common-mode output gain)

**PSRR+ / PSRR− — Instances `x3`, `x4`**
- Analysis: AC sweep
- Config:
    - `x3`: `vddpsrr` = DC + 1V AC on the positive rail
    - `x4`: `gndpsrr` = DC + 1V AC on the negative rail
- Measurements: `DCPSRp = vdb(ppsr1) @ 0.1 Hz`, `DCPSRn = vdb(npsr1) @ 0.1 Hz`

**Input/Output Noise Density + Integrated Noise — Instance `x8`**
- **Analysis (Pass 1):** `noise V(nsopout) Vn_in dec 10 0.1 1G` for spot densities
    - `input_noise_density_1Hz`, `input_noise_density_spot` (at `fspot = 10 kHz`)
    - `output_noise_density_1Hz`, `output_noise_density_spot`
- **Analysis (Pass 2):** Second noise run integrated from `fstart` to `BW = GBW / DC_gain_linear` (the open-loop −3 dB frequency) → gives `integrated_input_noise`, `integrated_output_noise` over the amplifier's gain-flat band
- **Swap-detection logic:** Handles the ngspice version-dependent swap between `inoise` / `onoise` labels by comparing magnitudes at `f=0`

**Slew Rate (SR+ / SR−) — Instance `x6`**
- Analysis: `tran 1n 20u` with a `PULSE` stimulus of amplitude `sr_vstep = 0.5V` around `VCM`
- **Measurement (rising):** Uses **10%–60%** of the settled swing to avoid small-signal settling region
    - `SR+ = (V60% − V10%) / (t60% − t10%)`
- **Measurement (falling):** Uses **90%–40%** of swing, with a time delay past `0.8 × pulse_width` to skip rising-edge ringing
    - `SR− = (V40% − V90%) / (t40% − t90%)`
- **Reported separately** (not as `min(|SR+|, |SR−|)`)

**Output Swing — Instance `x7`**
- Analysis: `dc Vswin 0 VDD 0.001` (1 mV step DC sweep)
- Config: Unity-gain feedback, DUT tracks input
- **Measurement — tracking criterion:** `|V(swout) − V(swin)| < 10 mV`
    - `vout_low = V(swout) when err=0.01 FALL=1`
    - `vout_high = V(swout) when err=0.01 RISE=LAST`
    - `output_swing = vout_high − vout_low`

**VOS Mismatch (Monte Carlo) — Instance `x5`**
- Analysis: DC operating-point analysis in a 50-run MC loop
- **Three-step process:**
    1. Baseline `op` with `mc_mm_switch=0` → captures systematic offset `vos_systematic`
    2. Set `mc_mm_switch=1`, repeat 50: `reset; op` → `AGAUSS` random variables re-sampled each run; accumulate `mc_vos_mm = (vout − ideal_VCM) − vos_systematic`
    3. Compute sample statistics: `σ = sqrt(var)`, report `vos_mismatch_3sigma = 3σ`

**Power, VOS25, Temperature Coefficient — Instance `x5`**
- Analysis: `dc temp −40 125 1` (DC parametric sweep over 166 temperatures)
- Measurements:
    - `Power = |I(VVDD) @ user_temp| × VDD` (user-specified temp, default 27°C)
    - `vos25 = V(vout) @ user_temp − VCM_ideal`
    - `TC = (V_pp / V_avg) / 165` (fractional variation per °C across −40°C to 125°C)

### 7. Key Strength #4 — PVT Coverage and Temperature Interpolation

Comprehensive coverage across process, voltage, and temperature:

| Dimension | Values |
|-----------|--------|
| Process corners (all supported) | `tt`, `ff`, `ss`, `fs`, `sf` |
| Reference temperatures | −40°C, 25°C, 85°C (full LUT characterization) |
| Other temperatures | First-order linear interpolation between the two bracketing reference points (extrapolation is rejected) |
| Supply voltage | Configurable per simulation |
| Load capacitance | Configurable per simulation |

**Extreme PVT Check (optional post-sizing stage)**

After convergence, the agent can re-simulate with the sized devices at worst-case corners — typically **SS / 85°C** (slow, hot) and **FF / −40°C** (fast, cold) — to verify robustness. Results are reported alongside nominal specs in the final report.

### 8. Key Strength #5 — Post-LLM Numerical Optimization (CMA-ES)

After the LLM converges on a working design, an optional CMA-ES optimization stage can further refine it. This moves beyond the discrete LUT grid to continuous-space fine tuning.

| Property | Value |
|----------|-------|
| Algorithm | CMA-ES with coordinate-descent warmup |
| Tunable variables | `*_L`, `*_WL_ratio`, `*_M`, `C1_value` (Cc), `Rc_value`, `ibias` |
| Objective | User-weighted combination of `power ↓`, `gain ↑`, `GBW ↑` (linear scale, not dB) |
| Population | λ = 16, max 20 generations, early-stop after 5 stagnant generations |
| Constraint handling | Penalty-based, with 5× weight on already-passing specs to prevent regression |
| Evaluation | Parallel batch via `ThreadPoolExecutor`, leveraging CircuitCollector's Redis+SQLite cache |
| Mirror groups | Preserved so mirrored devices stay matched |

**Typical outcome:** 10–30% power reduction or 1–3 dB extra gain margin beyond the LLM's final solution.

### 9. Design Flow (6 Stages)

```
[1] Spec Understanding ──▶ [2] Circuit Understanding ──▶ [3] Topology-Specific Sizing
                                                                    │
                                                          ┌─────────▼──────────┐
                                                          │  [4] Simulate &    │
                                                          │      Verify        │◀─┐
                                                          └─────────┬──────────┘  │
                                                             PASS?  │    NO       │
                                                                    │             │
                                                                   YES    ──▶ [5] Root-Cause
                                                                    ▼       Diagnosis
                                                    [6] Design Review
                                                    (Report + Extreme PVT + CMA-ES Optimize)
```

| Stage | What Happens |
|-------|--------------|
| [1] | Validate required specs (VDD, CL, Gain, GBW, PM); apply defaults for temp, corner, VSS; check LUT temperature availability |
| [2] | Parse netlist → match topology (5T OTA or TSM) → assign device roles → detect sub-block variants (single/cascode/lv_cascode) → auto-generate Jinja2 + TOML → register with CircuitCollector |
| [3] | gm/ID sizing per role: determine `gm` from `GBW/CL` → pick `gm/ID` operating point → sweep `L` for gain → derive `W`, `M`, and all parasitics from LUT → analytical self-check |
| [4] | Run SPICE → verify saturation margins → compare specs against targets → decide PASS / iterate / timeout |
| [5] | Topology-specific fault tree analysis → targeted parameter adjustments → return to [3] |
| [6] | Final spec compliance table; optional Extreme PVT (SS/85°C, FF/−40°C); optional CMA-ES refinement |

**Iteration limits:** max 10 sizing-simulate-diagnose loops; max 30 total agent turns.

### 10. Spec Form Interface

Defined in `spec-form-template.md`.

**Required** (sizing will not proceed without all five):
- `VDD`, `CL`, `Gain` (dB), `GBW` (Hz), `PM` (degrees)

**Optional** (leave blank to skip measurement):
- `Power`, `SR+`, `SR−` (reported separately, never as `min`), `CMRR`, `PSRR+`, `PSRR−`, `IRN`, `ORN`, `Output_swing`, `I_bias`, `Mismatch`

**Post-sizing options:**
- `Extreme_PVT: yes/no` — run SS/85°C + FF/−40°C verification after convergence
- `Optimize: yes/no` — run CMA-ES after convergence

**Environment defaults (silent):**
- Temperature = 27°C (linear-interpolated against 25°C/85°C LUTs)
- Corner = `tt`, VSS = 0V

### 11. Technology Support

| Property | Value |
|----------|-------|
| PDK | SkyWater SKY130 (130 nm) |
| Devices | `sky130_fd_pr__nfet_01v8`, `sky130_fd_pr__pfet_01v8` |
| Nominal supply | 1.8 V |
| Simulator | ngspice v42+ |
| Channel lengths | 13 values: 0.18, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0 µm |

The architecture is **PDK-agnostic** — adding a new technology requires generating LUT data and providing PDK library paths to CircuitCollector.

### 12. Summary of Key Differentiators

1. **Sub-block black-box abstraction** — active loads and current mirrors expose a 5-quantity interface (`gds_eq`, `C_eq`, `p_int`, `V_headroom`, `Vbias_ext`), so single / cascode / LV-cascode variants plug into the same equations. This yields **9 structural variants per topology** with zero equation duplication.
2. **Physics-level equations + parasitic-aware LUTs** — per-width capacitances (`Cgg/W`, `Cgd/W`, `Cgs/W`, `Cdb/W`) scale into explicit pole/zero expressions, so the agent predicts behavior **before** simulation and diagnoses root causes **after**.
3. **Carefully engineered testbenches** — dedicated instances per spec, zero-plateau-corrected GBW, 10%–60% slew-rate window, closed-loop-bandwidth integrated noise, systematic-subtracted MC mismatch.
4. **Full PVT coverage** — 5 corners × 3 reference temps + linear interpolation + optional extreme-PVT re-verification stage.
5. **Post-LLM CMA-ES optimization** — continuous-space refinement beyond the discrete LUT grid, with constraint-preserving penalties.

---

## Appendix B: Historical Run Log — `tsm_single`

Verbatim Claude Code session (encoding artifacts cleaned) sizing `examples/tsm_variants/tsm_single.sp` against `spec-form-template.md`. Target specs: VDD=1.8 V, CL=5 pF, Gain>70 dB, GBW>50 MHz, PM>50°, ff corner @ 20°C, with mismatch, extreme PVT, and CMA-ES optimization all enabled.

```
use analog amplifier skill size tsm_single.sp     use the specs in the template
analog-amplifier skill
I'll follow the analog amplifier design flow. Let me start by reading the spec form and netlist, plus the first skill file.

Read spec-form-template.md
Read tsm_single.sp
Read spec-understanding.md
The netlist file wasn't found at the root. Let me search for it.

Globpattern: "**/*tsm_single*"
Found 2 files
Read tsm_single.sp
Now let me check LUT temperature availability and validate the specs.

Bash Check LUT temperature availability for ff corner at 20C
IN
python3 -c "
from scripts.lut_lookup import list_available_L, _resolve_device, _discover_temps
temp_str = '20C'
corner = 'ff'
temps = _discover_temps(_resolve_device('nfet'), corner)
print('Available temps for nfet/ff:', temps)
temps_p = _discover_temps(_resolve_device('pfet'), corner)
print('Available temps for pfet/ff:', temps_p)
"

OUT
Available temps for nfet/ff: [-40, 25, 85]
Available temps for pfet/ff: [-40, 25, 85]
Update Todos

Validate spec form (Stage 1)

Parse netlist and identify topology (Stage 2)

Execute TSM design flow — initial sizing (Stage 3)

Simulate and verify (Stage 4)

Root-cause diagnosis loop if needed (Stage 5)

Design review (Stage 6)

Extreme PVT simulations

Numerical optimization
Stage 1 — Spec Understanding
Step 2 — Required Fields Check

REQUIRED FIELDS CHECK
======================
VDD  : 1.8 V       ✓
CL   : 5 pF        ✓
Gain : 70 dB       ✓
GBW  : 50 MHz      ✓
PM   : 50°         ✓

Status: ALL PRESENT → proceed
Step 3 — Environment
Temperature = 20°C (user-specified)
Corner = ff (user-specified)
VSS = 0 V (default)
temp_str = "20C", temperature = 20 (numeric for simulator)
LUT temperature availability: Available reference temps are [-40, 25, 85]°C. 20°C is between -40°C and 25°C.

LUT data at 20°C not pre-generated. Using first-order linear interpolation between -40°C and 25°C reference data. Note: this is an approximation; accuracy degrades far from reference points.

Step 4 — Optional Specs / Mismatch / Post-Sizing
Gain 70 dB = 10^(70/20) = 3162.3 V/V (linear)

VALIDATED SPECIFICATIONS
========================
Required:
  VDD  = 1.8 V
  CL   = 5e-12 F
  Gain > 70 dB  (3162.3 V/V)
  GBW  > 50 MHz
  PM   > 50°

Environment:
  Temperature = 20°C
  Corner      = ff
  VSS         = 0 V

Active Targets (will be optimized):
  Power        < 500 µW
  SR+          > 20 V/µs
  CMRR         > 60 dB
  PSRR+        > 60 dB
  PSRR-        > 60 dB
  IRN          < 30 µV rms
  Output_swing > 1.2 V
  I_bias       = 10 µA

Inactive (blank — will report achieved value only):
  SR-, ORN

Mismatch:
  Mismatch simulation     : enabled (15 mV, 3σ)

Post-Sizing:
  Extreme PVT check       : enabled
  Numerical optimization  : enabled
Now proceeding to Stage 2 — Circuit Understanding.

Read circuit-understanding.md
Stage 2 — Circuit Understanding
Step 1 — Parse Netlist & Identify Topology
Device inventory: 3 PMOS (M1, M2, M7), 5 NMOS (M3, M4, M5, M6, M8), 2 passives (Rc, C1)

Connections:

Device	D	G	S	B	Type
XM1	net1	net1	vdda	vdda	PMOS (diode)
XM2	net5	net1	vdda	vdda	PMOS
XM3	net1	vinn	net2	gnda	NMOS
XM4	net5	vinp	net2	gnda	NMOS
XM5	net3	net3	gnda	gnda	NMOS (diode)
XM6	net2	net3	gnda	gnda	NMOS
XM7	vout	net5	vdda	vdda	PMOS
XM8	vout	net3	gnda	gnda	NMOS
Topology match: Diff pair (M3/M4) + single-PMOS mirror load (M1/M2) + CS stage (M7) + Miller comp (Rc, C1) → Two-Stage Miller (TSM)

Step 2 — Role Assignment

Role	Primary	Mirrors	Type	mirror_of
LOAD	M1	[M2]	pfet	—
DIFF_PAIR	M3	[M4]	nfet	—
BIAS_GEN	M5	[]	nfet	—
TAIL	M6	[]	nfet	BIAS_GEN
OUTPUT_CS	M7	[]	pfet	—
OUTPUT_BIAS	M8	[]	nfet	BIAS_GEN

Step 2b — Sub-block detection: All roles are single (no cascode companions found).
Step 5 — Register Topology
Bash Register tsm_single topology with CircuitCollector
IN
python3 -c "
from tools import ensure_topology_registered

parameterized_netlist = '''
.subckt {{netlist_name}} gnda vdda vinn vinp vout Ib
* PMOS mirror load - single transistor per branch
XM1 net1  net1 vdda vdda sky130_fd_pr__pfet_01v8 l={{ M1_L }} w={{ M1_W }} m={{ M1_M }}
XM2 net5  net1 vdda vdda sky130_fd_pr__pfet_01v8 l={{ M1_L }} w={{ M1_W }} m={{ M1_M }}
* NMOS diff pair
XM3 net1  vinn net2 gnda sky130_fd_pr__nfet_01v8 l={{ M3_L }} w={{ M3_W }} m={{ M3_M }}
XM4 net5  vinp net2 gnda sky130_fd_pr__nfet_01v8 l={{ M3_L }} w={{ M3_W }} m={{ M3_M }}
* Tail current source + bias reference
XM5 net3  net3 gnda gnda sky130_fd_pr__nfet_01v8 l={{ M5_L }} w={{ M5_W }} m={{ M5_M }}
XM6 net2  net3 gnda gnda sky130_fd_pr__nfet_01v8 l={{ M5_L }} w={{ M5_W }} m={{ M6_M }}
* Second stage - PMOS CS amplifier + NMOS current source
XM7 vout  net5 vdda vdda sky130_fd_pr__pfet_01v8 l={{ M7_L }} w={{ M7_W }} m={{ M7_M }}
XM8 vout  net3 gnda gnda sky130_fd_pr__nfet_01v8 l={{ M5_L }} w={{ M5_W }} m={{ M8_M }}
* Miller compensation
R_Rc net5 net_rc {{ Rc_value }}
C_C1 net_rc vout {{ C1_value }}
I0  vdda  net3 Ib
.ends {{netlist_name}}
'''.strip()

role_device_map = {
    'LOAD':        {'primary': 'M1', 'mirrors': ['M2'], 'device_type': 'pfet', 'sub_block_type': 'single'},
    'DIFF_PAIR':   {'primary': 'M3', 'mirrors': ['M4'], 'device_type': 'nfet'},
    'BIAS_GEN':    {'primary': 'M5', 'mirrors': [],     'device_type': 'nfet', 'sub_block_type': 'single'},
    'TAIL':        {'primary': 'M6', 'mirrors': [],     'device_type': 'nfet', 'mirror_of': 'BIAS_GEN', 'sub_block_type': 'single'},
    'OUTPUT_CS':   {'primary': 'M7', 'mirrors': [],     'device_type': 'pfet'},
    'OUTPUT_BIAS': {'primary': 'M8', 'mirrors': [],     'device_type': 'nfet', 'mirror_of': 'BIAS_GEN', 'sub_block_type': 'single'},
}

result = ensure_topology_registered(
    topology_name='tsm_single',
    raw_netlist=parameterized_netlist,
    role_device_map=role_device_map,
    requires_Cc=True,
    passive_params=['C1_value', 'Rc_value'],
    extra_ports=[],
)
print(result)
"

OUT
{'status': 'error', 'message': "extra_ports must be a dict mapping port_name -> DC voltage (e.g. {'Vbias_cas_p': 0.6, 'Vbias_cas_n': 0.6}); got list."}

Bash Register tsm_single topology (fixed extra_ports to dict)
— same call with extra_ports={} —

OUT
{'status': 'ok', 'config_path': 'config/skywater/opamp/tsm_single.toml', 'registered': False}

CIRCUIT IDENTIFICATION
=======================
Topology      : Two-Stage Miller (TSM) — single-transistor PMOS mirror load
Match         : circuit-specific/tsm/
Devices       : 5 NMOS, 3 PMOS, 2 passives (Rc, C1)
Registration  : already_exists
Config path   : config/skywater/opamp/tsm_single.toml

Role-Device Map:
  LOAD        → M1 [+ M2] (pfet)
  DIFF_PAIR   → M3 [+ M4] (nfet)
  BIAS_GEN    → M5 (nfet)
  TAIL        → M6 (nfet) mirrors BIAS_GEN
  OUTPUT_CS   → M7 (pfet)
  OUTPUT_BIAS → M8 (nfet) mirrors BIAS_GEN

Sub-block structures:
  LOAD        : single
  BIAS_GEN    : single
  TAIL        : single
  OUTPUT_BIAS : single

Extra subcircuit ports: none
Passive params: C1_value, Rc_value

Activated design flow: circuit-specific/tsm/tsm-design-flow.md
Now proceeding to Stage 3 — TSM Design Flow.

Read tsm-design-flow.md
Now I have the full design flow. Let me execute the sizing procedure in Python.

Bash Execute full TSM sizing Steps 1-6 with LUT queries
IN
python3 << 'PYEOF2'
import numpy as np
from scripts.lut_lookup import lut_query, list_available_L

# Design parameters from validated spec form
VDD = 1.8
CL = 5e-12
Gain_dB = 70
A0_target = 10**(70/20)  # 3162.3 V/V
GBW_target = 50e6
PM_target = 50
I_bias = 10e-6
corner = 'ff'
temp = '20C'

# Step 1: DIFF_PAIR
Cc_initial = 0.5 * CL  # 2.5 pF
gm3 = 2*np.pi*GBW_target*Cc_initial
gm_id_3 = 12

# sweep L for gain requirement
sqrt_A0 = np.sqrt(A0_target)  # 56.2
available_L_nfet = list_available_L('nfet', corner, temp)
L3 = None
for L in available_L_nfet:
    gm_gds = lut_query('nfet', 'gm_gds', L, gm_id_val=gm_id_3, corner=corner, temp=temp)
    if gm_gds / 2 >= sqrt_A0:
        L3 = L; break

ID3 = gm3 / gm_id_3
I_tail = 2*ID3
# ... derive W3, gds3, Cgs3, Cgd3, Cdb3, vdsat3 from LUT
# (full derivation in script)

# Step 2: LOAD (single sub-block, L sweep for A_v1 >= sqrt_A0)
# Step 3: OUTPUT_CS (L7=0.18 shortest for speed)
# Step 4: BIAS mirrors (M5_M=1, M6_M=round(I_tail/I_bias), M8_M=round(ID7/I_bias))
# Step 5: Cc, Rc
# Step 6: analytical spec evaluation
PYEOF2

OUT  (analytical iteration 1)
A0_target (linear) = 3162.3 V/V
sqrt(A0_target) = 56.2 V/V  (per-stage gain target)

===== Step 1: DIFF_PAIR (M3/M4) =====
Cc_initial = 2.50 pF
gm3 = 2π × 50.0 MHz × 2.5 pF = 785.40 µS
Available L (nfet): [0.18, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]

Sweeping L for DIFF_PAIR (need gm_gds/2 >= 56.2):
  L=0.18 µm: gm_gds =  17.5, gm_gds/2 =   8.8 ❌
  L=0.50 µm: gm_gds =  92.9, gm_gds/2 =  46.5 ❌
  L=1.00 µm: gm_gds = 106.6, gm_gds/2 =  53.3 ❌
  L=1.50 µm: gm_gds = 144.3, gm_gds/2 =  72.2 ✓ sufficient
  L=2.00 µm: gm_gds = 165.3, gm_gds/2 =  82.7 ✓ sufficient
  L=2.50 µm: gm_gds = 191.1, gm_gds/2 =  95.6 ✓ sufficient
  L=3.00 µm: gm_gds = 213.2, gm_gds/2 = 106.6 ✓ sufficient
  L=3.50 µm: gm_gds = 232.5, gm_gds/2 = 116.2 ✓ sufficient
  L=4.00 µm: gm_gds = 249.4, gm_gds/2 = 124.7 ✓ sufficient
  L=4.50 µm: gm_gds = 230.6, gm_gds/2 = 115.3 ✓ sufficient
  L=5.00 µm: gm_gds = 307.0, gm_gds/2 = 153.5 ✓ sufficient
  L=5.50 µm: gm_gds = 332.2, gm_gds/2 = 166.1 ✓ sufficient
  L=6.00 µm: gm_gds = 261.9, gm_gds/2 = 131.0 ✓ sufficient

Selected L3 = 1.50 µm
gm/ID_3 = 12 S/A
ID3 = 65.45 µA, I_tail = 130.90 µA
id_w3 = 2.2786 A/m, W3 = 28.72 µm
gm3 = 785.40 µS, gds3 = 5.4414 µS
ft3 = 0.51 GHz
Cgs3 = 219.95 fF, Cgd3 = 0.34 fF, Cdb3 = 20.80 fF
vdsat3 = 145.9 mV

===== Step 2: LOAD (M1/M2) =====
Available L (pfet): [0.18, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]

Sweeping L for LOAD (single, need A_v1 >= 56.2):
  L=0.18 µm: gds1 = 57.3538 µS, A_v1 =  12.5 ❌
  L=0.50 µm: gds1 =  8.5666 µS, A_v1 =  56.1 ❌
  L=1.00 µm: gds1 =  3.0470 µS, A_v1 =  92.5 ✓
  L=1.50 µm: gds1 =  1.9548 µS, A_v1 = 106.2 ✓
  L=2.00 µm: gds1 =  1.4504 µS, A_v1 = 114.0 ✓
  L=2.50 µm: gds1 =  1.2115 µS, A_v1 = 118.1 ✓
  L=3.00 µm: gds1 =  1.0540 µS, A_v1 = 120.9 ✓
  L=3.50 µm: gds1 =  0.9423 µS, A_v1 = 123.0 ✓
  L=4.00 µm: gds1 =  0.8590 µS, A_v1 = 124.7 ✓
  L=4.50 µm: gds1 =  0.7232 µS, A_v1 = 127.4 ✓
  L=5.00 µm: gds1 =  0.6280 µS, A_v1 = 129.4 ✓
  L=5.50 µm: gds1 =  0.5584 µS, A_v1 = 130.9 ✓
  L=6.00 µm: gds1 =  0.4235 µS, A_v1 = 133.9 ✓

Selected L1 = 1.00 µm
gm/ID_1 = 12 S/A
ID1 = 65.45 µA
gm1 = 785.40 µS, gds1 = 3.0470 µS
id_w1 = 0.8233 A/m, W1 = 79.50 µm
ft1 = 0.29 GHz
Cgs1 = 374.09 fF, Cgd1 = 0.35 fF, Cdb1 = 34.53 fF
vdsat1 = 146.0 mV

A_v1 = gm3 / (gds3 + gds1) = 92.5

===== Step 3: OUTPUT_CS (M7) =====
ω_c = 50.00 MHz  (rad: 3.14e+08 rad/s)
gm7_required (PM) = 2.2 × ω_c × CL = 3455.75 µS
gm/ID_7 = 10 S/A
ID7_from_PM = 345.58 µA
ID7 = 345.58 µA

Trying L7 = 0.18 µm (shortest for speed)
gm7 = 3455.75 µS
id_w7 = 5.8200 A/m, W7 = 59.38 µm
gds7 = 263.9312 µS
ft7 = 9.06 GHz
Cgs7 = 42.65 fF, Cgd7 = 1.85 fF, Cdb7 = 3.76 fF
vdsat7 = 167.0 mV

===== Step 4: BIAS_GEN, TAIL, OUTPUT_BIAS =====
L5 = 1.00 µm, gm/ID_5 = 12 S/A
M5_M = 1, M6_M = 13, M8_M = 35
⚠ High mirror ratio: max = 35 → risk of VDS compression
W5 = 3.01 µm, gm_finger = 120.00 µS, gds_finger = 1.1256 µS
vdsat5 = 142.7 mV

M6 (TAIL): gm6 = 1560.00 µS, gds6 = 14.6323 µS
M8 (OUTPUT_BIAS): gm8 = 4200.00 µS, gds8 = 39.3947 µS

===== Step 5: Compensation =====
Cc = gm3 / (2π × GBW) = 2.50 pF
p3 (mirror pole) =  0.17 GHz
p4 (net5 pole)   = 12.69 GHz
PM_est = 73.1°

Rc = (1/gm7) × (1 + CL/Cc) = 868.1 Ω

===== Step 6: Analytical Spec Evaluation =====
A_v1 = 92.5, A_v2 = 11.4
A0 = 1054.1 (60.5 dB)
GBW = 50.00 MHz
PM = 73.1°
SR+ = 52.36 V/µs
SR- = 46.06 V/µs
Output swing = 1.490 V
Power = 875.65 µW
CMRR = 79.9 dB
PSRR- = 76.0 dB
PSRR+ = 59.9 dB

Spec               Analytical       Target   Status
----------------------------------------------------
A0 (dB)                 60.46        70.00        ❌
GBW (MHz)               50.00        50.00        ✓
PM (°)                  73.11        50.00        ✓
Power (µW)             875.65       500.00        ❌
SR+ (V/µs)              52.36        20.00        ✓
CMRR (dB)               79.94        60.00        ✓
PSRR+ (dB)              59.88        60.00        ❌
PSRR- (dB)              75.95        60.00        ✓
Swing (V)                1.49         1.20        ✓

All specs met: False

===== SIZING SUMMARY =====
M3/M4 (DIFF_PAIR):  L=1.50µm,  W=28.72µm,  M=1,   gm/ID=12
M1/M2 (LOAD):       L=1.00µm,  W=79.50µm,  M=1,   gm/ID=12
M7    (OUTPUT_CS):  L=0.18µm,  W=59.38µm,  M=1,   gm/ID=10
M5    (BIAS_GEN):   L=1.00µm,  W=3.01µm,   M=1,   gm/ID=12
M6    (TAIL):       L=1.00µm,  W=3.01µm,   M=13,  (mirrors M5)
M8    (OUTPUT_BIAS):L=1.00µm,  W=3.01µm,   M=35,  (mirrors M5)
Cc = 2.50 pF, Rc = 868.1 Ω
I_bias = 10.0 µA, I_tail = 130.90 µA, ID7 = 345.58 µA

Three specs failed: A0 (60.5 vs 70 dB), Power (876 vs 500 µW), and PSRR+ (59.9 vs 60 dB). Let me read the root-cause diagnosis and apply fixes.

Read tsm-root-cause-diagnosis.md
Root cause analysis:
  • A0 low: A_v2 = 11.4 because L7 = 0.18 µm gives very low gm_gds → increase L7
  • Power high: ID7 = 346 µA is excessive (PM already 73° >> 50°) → increase gm/ID_7, reduce gm7 target
  • PSRR+: borderline, will improve with gain fix
Applying fixes: L7 → 1.0 µm, gm/ID_7 → 12, relaxed gm7 target.

Bash Execute analytical iteration 2 with fixed L7 and gm/ID_7
OUT
PM budget: 90 - 16.7° (p3) - pm_p4 >= 55°
max_pm_from_p4 = 18.3°
p4_min = 150.8 MHz, gm7_min = 47.4 µS
gm7 from p2 > 2×ωc×CL = 3141.6 µS

gm7_required = 3141.6 µS, ID7 = 261.80 µA
Power check: 724.9 µW (target < 500 µW)
ID7_max for 500µW = 136.88 µA
Adjusted: ID7 = 136.88 µA, gm7 = 1642.5 µS

M7: L=1.0µm, W=166.26µm, gm7=1642.5µS, gds7=6.3723µS
    ft7=0.29GHz, vdsat7=146.0mV

M5_M=1, M6_M=13, M8_M=14
⚠ High mirror ratio: max=14

Cc = 2.50 pF
SR+ = 52.36 V/µs (target: 20)
p3 = 0.17 GHz, p4 = 0.33 GHz
PM_est = 64.8°
Rc = 1826.4 Ω

===== ANALYTICAL SPEC CHECK (Iteration 2) =====
A_v1 = 92.5, A_v2 = 74.2

Spec               Analytical       Target   Status
----------------------------------------------------
A0 (dB)                 76.74        70.00        ✓
GBW (MHz)               50.00        50.00        ✓
PM (°)                  64.82        50.00        ✓
Power (µW)             500.00       500.00        ✓
SR+ (V/µs)              52.36        20.00        ✓
CMRR (dB)               79.94        60.00        ✓
PSRR+ (dB)              82.19        60.00        ✓
PSRR- (dB)              77.09        60.00        ✓
Swing (V)                1.51         1.20        ✓

All specs met: True

===== SIZING SUMMARY (Iteration 2) =====
M3/M4 (DIFF_PAIR):  L=1.5µm,  W=28.72µm,   M=1,   gm/ID=12
M1/M2 (LOAD):       L=1.0µm,  W=79.50µm,   M=1,   gm/ID=12
M7    (OUTPUT_CS):  L=1.0µm,  W=166.26µm,  M=1,   gm/ID=12
M5    (BIAS_GEN):   L=1.0µm,  W=3.01µm,    M=1,   gm/ID=12
M6    (TAIL):       L=1.0µm,  W=3.01µm,    M=13
M8    (OUTPUT_BIAS):L=1.0µm,  W=3.01µm,    M=14
Cc = 2.50 pF, Rc = 1826.4 Ω
I_bias = 10.0 µA, I_tail = 130.90 µA, ID7 = 136.88 µA

All analytical specs pass in iteration 2. Now proceeding to Step 7 — submit to simulation.

Bash Submit sizing to simulation (convert + simulate)
IN
python3 << 'PYEOF2'
from tools import convert_sizing, simulate_circuit

result = convert_sizing(
    topology='twostage',
    roles_raw={
        "DIFF_PAIR":    {"gm_id_target": 12, "L_guidance_um": 1.5, "id_derived": 6.545e-05},
        "LOAD":         {"gm_id_target": 12, "L_guidance_um": 1.0, "id_derived": 6.545e-05},
        "OUTPUT_CS":    {"gm_id_target": 12, "L_guidance_um": 1.0, "id_derived": 1.3688e-04},
        "BIAS_GEN":     {"gm_id_target": 12, "L_guidance_um": 1.0, "id_derived": 1e-05},
        "TAIL":         {"gm_id_target": 0,  "L_guidance_um": 1.0, "id_derived": 1.309e-04},
        "OUTPUT_BIAS":  {"gm_id_target": 0,  "L_guidance_um": 1.0, "id_derived": 1.3688e-04},
    },
    Ib_a=1e-05,
    Cc_f=2.5e-12,
    Rc_ohm=1826.4,
    l_overrides={"DIFF_PAIR": 1.5, "LOAD": 1.0, "OUTPUT_CS": 1.0,
                 "BIAS_GEN": 1.0, "TAIL": 1.0, "OUTPUT_BIAS": 1.0},
)
print("convert_sizing result:")
print(f"  config_path: {result['config_path']}")
# ...
PYEOF2

OUT
Exit code 1
Traceback (most recent call last):
  File "<stdin>", line 24, in <module>
KeyError: 'config_path'

Bash Debug convert_sizing output keys
OUT
Full result:
  status: error
  message: Unknown topology: 'twostage'. Available: []

Bash Convert sizing with correct topology name tsm_single
OUT
Full result:
  status: error
  message: Unknown topology: 'tsm_single'. Available: []

Grep "def convert_sizing" (in AnalogAgent)          → 2 lines of output
Read param_converter.py (lines 2-81)
Grep "def ensure_topology_registered" (in AnalogAgent) → 2 lines of output
Read topology_manager.py
The registry is in-memory — I need to call ensure_topology_registered in the same Python process before convert_sizing.

Bash Full simulation pipeline: register + convert + simulate
(combined ensure_topology_registered + convert_sizing + simulate_circuit in a single python3 <<PYEOF block)

OUT
Registration: {'status': 'ok', 'config_path': 'config/skywater/opamp/tsm_single.toml', 'registered': False}

convert_sizing result:
  status: ok
  params: {'M5_L': 1.0, 'M5_WL_ratio': 3.44, 'M5_M': 1, 'M6_L': 1.0, 'M6_WL_ratio': 3.44, 'M6_M': 13,
           'M8_L': 1.0, 'M8_WL_ratio': 3.44, 'M8_M': 14, 'M3_L': 1.5, 'M3_WL_ratio': 7.28, 'M3_M': 3,
           'M1_L': 1.0, 'M1_WL_ratio': 9.8, 'M1_M': 10, 'M7_L': 1.0, 'M7_WL_ratio': 9.76, 'M7_M': 21,
           'C1_value': 2.5e-12, 'Rc_value': 1826.4, 'ibias': 1e-05}
  config_path: config/skywater/opamp/tsm_single.toml

Running simulation...

===== SIMULATION RESULTS =====
Specs:
  dcgain_: 82.16144
  gain_bandwidth_product_: 45999800.0
  phase_margin: 49.85237
  cmrr: 71.36544
  dcpsrp: -76.8488
  dcpsrn: -69.4733
  power: 485.6355
  vos25: -0.0001027
  tc: 1.816059e-07
  input_noise_density_spot: 6.976257e-08
  output_noise_density_spot: 0.0003200507
  integrated_input_noise: 1.4823e-05
  integrated_output_noise: 0.18484
  slew_rate_pos: 73432820.0
  slew_rate_neg: -9104160.0
  vout_low: 0.1060801
  vout_high: 1.436505
  output_swing: 1.330425
  gain_peaking_db: 0.0
  true_gbw: 45999800.0
  vos_mismatch_3sigma: 0.009170673

Transistor OP data:
Traceback ...  AttributeError: 'TransistorOP' object has no attribute 'get'
(inspected the dataclass; TransistorOP has attrs: name, gm, gds, id, vgs, vds, vth, region, cgg)

Bash Full simulation results with specs comparison (SPICE iteration 1)
OUT
Transistor OP data:
Name            Vds      Vgs      Vth     Id(µA)     gm(µS)    gds(µS)       region
--------------------------------------------------------------------------------
M1           1.1006   1.1006   1.0150      58.71     768.76     1.6650   saturation
M2           1.1126   1.1006   1.0150      58.73     769.01     1.6524   saturation
M3           0.5160   0.7165   0.5889      58.71     798.67     5.1346   saturation
M4           0.5040   0.7166   0.5889      58.73     798.61     5.2295   saturation
M5           0.7069   0.7069   0.5853      10.00     125.91     0.9381   saturation
M6           0.1834   0.7069   0.5853     117.44    1443.46    88.0716   saturation
M7           0.9001   1.1126   1.0150     142.35    1782.66     4.6658   saturation
M8           0.8999   0.7069   0.5853     142.35    1787.31    11.4436   saturation

Spec                                  SPICE       Target   Status
-----------------------------------------------------------------
DC Gain (dB)                         82.161       70.000        ✓
GBW (MHz)                            46.000       50.000        ❌
Phase Margin (°)                     49.852       50.000        ❌
Power (µW)                          485.635      500.000        ✓
SR+ (V/µs)                           73.433       20.000        ✓
CMRR (dB)                            71.365       60.000        ✓
PSRR+ (dB)                           76.849       60.000        ✓
PSRR- (dB)                           69.473       60.000        ✓
IRN (µV rms)                         14.823       30.000        ✓
Output Swing (V)                      1.330        1.200        ✓
Mismatch 3σ (mV)                      9.171       15.000        ✓

All specs met: False

Simulation results show:
  • DC Gain = 82.2 dB ✓, Power = 485.6 µW ✓
  • GBW = 46.0 MHz ❌ (target 50), PM = 49.85° ❌ (target 50 — borderline)
  • All transistors in saturation ✓, but M6 has tight Vds margin (183 mV, gds=88 µS)
Root cause: GBW 8% low due to parasitic loading; PM 0.15° short. Fix: reduce Cc by ~10% to boost GBW, adjust Rc.

Bash Simulation iteration 2: reduced Cc, adjusted Rc
(Cc = 2.20 pF, Rc_new = (1/gm7_spice)(1 + CL/Cc_new) = 1835.5 Ω using SPICE gm7=1783µS)

OUT
Spec                                  SPICE       Target   Status
-----------------------------------------------------------------
DC Gain (dB)                         82.161       70.000        ✓
GBW (MHz)                            47.617       50.000        ❌
Phase Margin (°)                     44.316       50.000        ❌
Power (µW)                          485.635      500.000        ✓
…
Reducing Cc hurt PM significantly. The issue is the Rc was computed with analytical gm7 (1642 µS) but SPICE gm7=1783 µS. Go back to Cc=2.5pF with corrected Rc, then try Cc=2.35pF for GBW.

Bash Simulation iteration 3: Cc=2.35pF with corrected Rc from SPICE gm7
OUT
Iteration 3: Cc = 2.35 pF, Rc = 1754.2 Ω
DC Gain 82.2 dB ✓, GBW=45.7 MHz ❌, PM=47.1° ❌  (worse than Cc=2.5 baseline)

Reducing Cc makes it worse — the issue is parasitic loading, not Cc. Let me try a structural fix: trade excess gain (82 vs 70 dB) for bandwidth by shortening L3 and L7.

Bash Simulation iteration 3b: L3=1.0, L7=0.5 for more bandwidth
(kept gm/ID all 12; M7_M recomputed, M3/L shorter)

OUT
gm3=785.4µS, ID3=65.45µA, I_tail=130.90µA
ID7=136.88µA, gm7_est=1642.5µS
M6_M=13, M8_M=14
Cc=2.50pF, Rc=1826.4Ω

A_v1=75.4, A_v2=48.8, A0=71.3 dB

Spec                                  SPICE       Target   Status
-----------------------------------------------------------------
DC Gain (dB)                         77.706       70.000        ✓
GBW (MHz)                            49.481       50.000        ❌  (0.5 MHz short)
Phase Margin (°)                     72.116       50.000        ✓
Power (µW)                          483.300      500.000        ✓
SR+ (V/µs)                           90.812       20.000        ✓
CMRR (dB)                            60.949       60.000        ✓
PSRR+ (dB)                           69.974       60.000        ✓
PSRR- (dB)                           66.952       60.000        ✓
IRN (µV rms)                         22.656       30.000        ✓
Output Swing (V)                      1.609        1.200        ✓
Mismatch 3σ (mV)                      9.397       15.000        ✓

Transistor OP:
  M1: Vds=1.0998 Id=58.11µA gm=763.0µS gds=1.65µS saturation
  M2: Vds=1.0555 Id=58.03µA gm=762.1µS gds=1.70µS saturation
  M3: Vds=0.5300 Id=58.11µA gm=787.8µS gds=5.44µS saturation
  M4: Vds=0.5744 Id=58.03µA gm=787.9µS gds=5.12µS saturation
  M5: Vds=0.7069 Id=10.00µA gm=125.9µS gds=0.94µS saturation
  M6: Vds=0.1702 Id=116.14µA gm=1414.5µS gds=110.68µS saturation
  M7: Vds=0.8996 Id=142.36µA gm=1814.7µS gds=15.80µS saturation
  M8: Vds=0.9004 Id=142.36µA gm=1787.4µS gds=11.44µS saturation

Excellent progress! PM = 72° (huge margin), gain = 77.7 dB ✓, GBW = 49.48 MHz (only 0.5 MHz short). Since PM has 22° of margin, slightly reduce Cc to push GBW over 50 MHz.

Sweep Cc (2.45, 2.42, 2.38 pF) with same Rc to find GBW > 50 MHz
OUT
Cc=2.45pF Rc=1826Ω → Gain=77.7dB GBW=49.87MHz PM=71.31° Pwr=483.3µW
Cc=2.42pF Rc=1826Ω → Gain=77.7dB GBW=50.11MHz PM=70.81° Pwr=483.3µW  ← chosen
Cc=2.38pF Rc=1826Ω → Gain=77.7dB GBW=50.50MHz PM=70.11° Pwr=483.3µW

Final verification simulation with Cc=2.42pF, mismatch enabled
OUT
Spec                                  SPICE       Target   Status
-----------------------------------------------------------------
DC Gain (dB)                         77.706       70.000        ✓
GBW (MHz)                            50.111       50.000        ✓
Phase Margin (°)                     70.815       50.000        ✓
Power (µW)                          483.300      500.000        ✓
SR+ (V/µs)                           92.509       20.000        ✓
CMRR (dB)                            60.949       60.000        ✓
PSRR+ (dB)                           69.974       60.000        ✓
PSRR- (dB)                           66.952       60.000        ✓
IRN (µV rms)                         22.648       30.000        ✓
Output Swing (V)                      1.609        1.200        ✓
Mismatch 3σ (mV)                     12.512       15.000        ✓

All specs met: True

Transistor OP:
Name        Vds      Vgs      Vth     Id(µA)     gm(µS)    gds(µS)       region
M1       1.0998   1.0998   1.0150      58.11     763.05     1.6498   saturation
M2       1.0555   1.0998   1.0150      58.03     762.13     1.6983   saturation
M3       0.5300   0.7302   0.6137      58.11     787.83     5.4351   saturation
M4       0.5744   0.7298   0.6137      58.03     787.95     5.1219   saturation
M5       0.7069   0.7069   0.5853      10.00     125.91     0.9381   saturation
M6       0.1702   0.7069   0.5853     116.14    1414.52   110.6762   saturation
M7       0.8996   1.0555   0.9647     142.36    1814.66    15.8039   saturation
M8       0.9004   0.7069   0.5853     142.36    1787.36    11.4404   saturation

Final params:
  {'M5_L': 1.0, 'M5_WL_ratio': 3.44, 'M5_M': 1, 'M6_L': 1.0, 'M6_WL_ratio': 3.44, 'M6_M': 13,
   'M8_L': 1.0, 'M8_WL_ratio': 3.44, 'M8_M': 14, 'M3_L': 1.0, 'M3_WL_ratio': 7.51, 'M3_M': 3,
   'M1_L': 1.0, 'M1_WL_ratio': 9.8, 'M1_M': 10, 'M7_L': 0.5, 'M7_WL_ratio': 9.68, 'M7_M': 23,
   'C1_value': 2.42e-12, 'Rc_value': 1826.4, 'ibias': 1e-05}

All specs pass! All transistors in saturation. Now proceeding to the design review, extreme PVT, and optimization.

Read design-review.md

Bash Analytical re-derivation + extreme PVT simulations (SS/85°C, FF/-40°C)
OUT
ANALYTICAL (final sizes):
  A0=71.3 dB, GBW=51.7 MHz, PM=70.6°
  SR+=54.1 V/µs, SR-=18.4 V/µs
  Power=500.0 µW, Swing=1.511 V, CMRR=78.2 dB

--- SS/85°C ---
  dcgain_: 77.20043
  gain_bandwidth_product_: 35944700.0
  phase_margin: 66.26543
  cmrr: 55.04365
  dcpsrp: -71.2113
  dcpsrn: -57.353
  power: 469.3162
  vos25: 0.0003695
  tc: 1.798347e-06
  input_noise_density_spot: 1.11748e-07
  output_noise_density_spot: 0.0003910707
  integrated_input_noise: 2.38698e-05
  integrated_output_noise: 0.168972
  slew_rate_pos: 73071270.0
  slew_rate_neg: -10306600.0
  vout_low: 1.705949
  vout_high: 1.709118
  output_swing: 0.003169
  gain_peaking_db: 0.0
  true_gbw: 35944700.0
  vos_mismatch_3sigma: 0.01251216

Transistors (SS/85°C):
  M1: Vds=1.0669 Vth=0.9662 gds=1.51µS SAT
  M2: Vds=1.0267 Vth=0.9662 gds=1.56µS SAT
  M3: Vds=0.5811 Vth=0.5899 gds=4.38µS SAT
  M4: Vds=0.6213 Vth=0.5899 gds=4.15µS SAT
  M5: Vds=0.7384 Vth=0.5666 gds=0.83µS SAT
  M6: Vds=0.1520 Vth=0.5666 gds=248.13µS SAT
  M7: Vds=0.8996 Vth=0.9138 gds=12.69µS SAT
  M8: Vds=0.9004 Vth=0.5666 gds=10.20µS SAT

--- FF/-40°C ---
  dcgain_: 78.31841
  gain_bandwidth_product_: 64475190.0
  phase_margin: 71.96076
  cmrr: 61.83137
  dcpsrp: -71.0428
  dcpsrn: -67.7846
  power: 480.0681
  vos25: 0.0002962
  tc: 1.994543e-06
  input_noise_density_spot: 9.730611e-08
  output_noise_density_spot: 0.0004536104
  integrated_input_noise: 2.17722e-05
  integrated_output_noise: 0.173483
  slew_rate_pos: 118312400.0
  slew_rate_neg: -9394030.0
  vout_low: 0.2317086
  vout_high: 1.726936
  output_swing: 1.495227
  gain_peaking_db: 1.09741
  true_gbw: 56822730.0
  vos_mismatch_3sigma: 0.01251216

Transistors (FF/-40°C):
  M1: Vds=1.1881 Vth=1.1085 gds=1.60µS SAT
  M2: Vds=1.1543 Vth=1.1085 gds=1.63µS SAT
  M3: Vds=0.4646 Vth=0.6623 gds=6.34µS SAT
  M4: Vds=0.4985 Vth=0.6623 gds=6.02µS SAT
  M5: Vds=0.7333 Vth=0.6369 gds=0.99µS SAT
  M6: Vds=0.1473 Vth=0.6369 gds=122.72µS SAT
  M7: Vds=0.8997 Vth=1.0653 gds=17.13µS SAT
  M8: Vds=0.9003 Vth=0.6369 gds=12.56µS SAT

Good — PVT results collected. SS/85°C shows GBW and swing degradation (expected), FF/-40°C passes most specs. Now let me read the optimization skill and ask the user for their priority.

Read numerical-optimization.md

DESIGN REVIEW

1. OUTCOME

STATUS: SUCCESS ✓ all specs met in 5 simulation iterations
  (2 analytical iterations + 5 SPICE iterations)

2. SPECIFICATION COMPLIANCE

Spec	        Target	Analytical	SPICE	Error	Margin	Status
DC Gain	        ≥ 70 dB	 71.3 dB	77.7 dB	 +9.0%	 +7.7 dB	✓
GBW	        ≥ 50 MHz	51.7 MHz	50.1 MHz	−3.1%	+0.1 MHz	✓
PM	        ≥ 50°	 70.6°	 70.8°	 +0.3%	 +20.8°	✓
Power	        ≤ 500 µW	500.0 µW	483.3 µW	−3.3%	−16.7 µW	✓
SR+	        ≥ 20 V/µs	54.1 V/µs	92.5 V/µs	+70.9%	+72.5 V/µs	✓
SR-	        (inactive)	18.4 V/µs	10.2 V/µs	—	—	--
CMRR	        ≥ 60 dB	 78.2 dB	60.9 dB	−22.1%	+0.9 dB	✓
PSRR+	        ≥ 60 dB	—	70.0 dB	—	+10.0 dB	✓
PSRR-	        ≥ 60 dB	—	67.0 dB	—	+7.0 dB	✓
IRN	        ≤ 30 µV	—	22.6 µV	—	−7.4 µV	✓
Output Swing	≥ 1.2 V	 1.511 V	1.609 V	+6.5%	+0.409 V	✓
Mismatch 3σ	≤ 15 mV	—	12.5 mV	—	−2.5 mV	✓

3. SIZING SUMMARY

Topology : TSM Single (Two-Stage Miller, single PMOS mirror load)
Process  : SKY130 / ff / 20°C
VDD      : 1.8 V
CL       : 5 pF
I_bias   : 10 µA

Role          | Device | W(µm)     | L(µm) | M  | ID(µA)  | gm/ID | Vdsat(mV)
LOAD          | M1/M2  | 9.8       | 1.0   | 10 | 58.1    | 13.1  | 85
DIFF_PAIR     | M3/M4  | 7.51(×L)  | 1.0   | 3  | 58.1    | 13.6  | 117
BIAS_GEN      | M5     | 3.44(×L)  | 1.0   | 1  | 10.0    | 12.6  | 121
TAIL          | M6     | 3.44(×L)  | 1.0   | 13 | 116.1   | 12.2  | 121
OUTPUT_CS     | M7     | 9.68(×L)  | 0.5   | 23 | 142.4   | 12.7  | 91
OUTPUT_BIAS   | M8     | 3.44(×L)  | 1.0   | 14 | 142.4   | 12.6  | 121

CircuitCollector params:
  M3_L=1.0  M3_WL_ratio=7.51  M3_M=3
  M1_L=1.0  M1_WL_ratio=9.8   M1_M=10
  M5_L=1.0  M5_WL_ratio=3.44  M5_M=1
  M6_M=13   M8_M=14
  M7_L=0.5  M7_WL_ratio=9.68  M7_M=23
  C1_value=2.42pF  Rc_value=1826.4Ω  ibias=10µA

4. ITERATION HISTORY

Iter	Change Made	Key Results	Decision
1 (anal.)	L3=1.5, L1=1.0, L7=0.18, gm/ID all 12	A0=60.5dB, GBW=50, PM=73°	❌ Gain, Power, PSRR+
2 (anal.)	L7→1.0, ID7 capped at budget	        A0=76.7dB, GBW=50, PM=65°	✓ All analytical
3 (SPICE)	First sim, Cc=2.5pF	                A0=82.2, GBW=46, PM=49.9°	❌ GBW, PM
4 (SPICE)	L3→1.0, L7→0.5 (trade gain for BW)	A0=77.7, GBW=49.5, PM=72.1°	❌ GBW (0.5 MHz short)
5 (SPICE)	Cc→2.42pF (minor reduction)	        A0=77.7, GBW=50.1, PM=70.8°	✓ All specs pass

5. EXTREME PVT CHECK

Spec	        Target	Design (ff/20°C)	SS/85°C	FF/−40°C
DC Gain (dB)	≥ 70	77.7	77.2 ✓	78.3 ✓
GBW (MHz)	≥ 50	50.1	35.9 ❌	64.5 ✓
PM (°)	        ≥ 50	70.8	66.3 ✓	72.0 ✓
Power (µW)	≤ 500	483.3	469.3 ✓	480.1 ✓
SR+ (V/µs)	≥ 20	92.5	73.1 ✓	118.3 ✓
CMRR (dB)	≥ 60	60.9	55.0 ❌	61.8 ✓
PSRR+ (dB)	≥ 60	70.0	71.2 ✓	71.0 ✓
PSRR- (dB)	≥ 60	67.0	57.4 ❌	67.8 ✓
Output Swing (V) ≥ 1.2	1.61	0.003 ❌	1.50 ✓

OP Flags:
  SS/85°C:   All transistors saturated, but M6 Vds=152 mV with gds=248 µS (near triode boundary). Output stuck near VDD → swing collapses.
  FF/−40°C:  All transistors saturated. 1.1 dB gain peaking (acceptable).

Summary:
  SS/85°C:  5/9 active specs met. Critical failures: GBW (−28%), CMRR, PSRR-, Output Swing
  FF/−40°C: 9/9 active specs met ✓

Optimization is enabled. Which metric should be prioritized?
  1. Power — minimize power consumption (default)
  2. Gain  — maximize DC gain
  3. GBW   — maximize gain-bandwidth product
Enter 1, 2, or 3 (or press Enter for default):

User: 3

GBW priority selected: w_pwr=0.15, w_gain=0.15, w_gbw=1.0. Running the numerical optimizer now.

Bash Run CMA-ES optimizer with GBW priority (w_gbw=1.0)
(17 variables including *_L, *_WL_ratio, *_M, C1_value, Rc_value, ibias;
 6 integer dims; mirror_groups enforce M6_L=M5_L etc.;
 λ=16, max_gen=20, early-stop at 5 stagnant gens, 16 parallel workers)

OUT
Optimization variables: 17
Integer variables:       6
σ₀ = 0.0728
Weights: w_pwr=0.15, w_gain=0.15, w_gbw=1.0

Computing baseline cost...
Baseline cost: 7.450643

Running coordinate warmup...
  Warmup: x0 cost = 7.4506
  Evaluating 33 warmup probes...
  Warmup:           M5_L -10% → cost=-0.8128 (Δ=-8.2635) ✓
  Warmup:    M5_WL_ratio -10% → cost=-0.0621 (Δ=-7.5127) ✓
  Warmup:           M3_L +10% → cost= 1.5320 (Δ=-5.9187) ✓
  Warmup:    M3_WL_ratio +10% → cost= 0.9331 (Δ=-6.5176) ✓
  Warmup:    M3_WL_ratio -10% → cost= 7.2690 (Δ=-0.1817) ✓
  Warmup:           M3_M +10% → cost=-1.4939 (Δ=-8.9445) ✓   ← biggest improvement
  Warmup:           M1_L +10% → cost=-0.4487 (Δ=-7.8993) ✓
  Warmup:    M1_WL_ratio +10% → cost= 2.6034 (Δ=-4.8472) ✓
  Warmup:           M1_M +10% → cost=-0.8179 (Δ=-8.2685) ✓
  Warmup:           M7_L +10% → cost=-0.7492 (Δ=-8.1999) ✓
  Warmup:           M7_L -10% → cost= 2.9158 (Δ=-4.5348) ✓
  Warmup:    M7_WL_ratio -10% → cost=-0.9783 (Δ=-8.4289) ✓
  Warmup:           M7_M -10% → cost=-0.2595 (Δ=-7.7101) ✓
  Warmup: 13 improving directions out of 33 probes
Warmup done in 43.7s
  Improving points: 13
  f0 = 7.450643
  diag_scale: [1. 1. 0.3 0.3 0.3 1. 1. 1. 1. 1. 1. 1. 1. 1. 0.3 0.3 0.3]

Running CMA-ES optimizer...
  σ₀ = 0.3/√17 = 0.0728
  C₀ shaped: 11/17 active dims
  Gen 1/20: best=-1.4939, gen_best=-1.4939, σ=0.0622, evals=16, stag=0
  Gen 2/20: best=-1.4939, gen_best= 2.2025, σ=0.0587, evals=32, stag=1
  Gen 3/20: best=-1.4939, gen_best= 6.7354, σ=0.0566, evals=48, stag=2
  Gen 4/20: best=-1.4939, gen_best= 0.6266, σ=0.0545, evals=64, stag=3
  Gen 5/20: best=-1.4939, gen_best= 1.4067, σ=0.0547, evals=80, stag=4
  Gen 6/20: best=-1.4939, gen_best=-1.4677, σ=0.0531, evals=96, stag=5
  Early stop: 5 stagnant generations

CMA-ES done in 72.2s (1.2 min)
  Best cost: -1.493860
  Evaluations: 96
  Generations: 6

Optimized params:
  M3_M:        3 → 13     (+333.3%)
  M6_L:        1
  M6_WL_ratio: 3.44
  M8_L:        1
  M8_WL_ratio: 3.44
  (all other parameters unchanged)

Final verification simulation (with mismatch)...

Spec                                    LLM    Optimized       Target   Status
--------------------------------------------------------------------------------
DC Gain (dB)                         77.706       78.845       70.000        ✓ (+1.5%)
GBW (MHz)                            50.111       73.940       50.000        ✓ (+47.6%)
Phase Margin (°)                     70.815       56.836       50.000        ✓ (-19.7%)
Power (µW)                          483.300      491.943      500.000        ✓ (+1.8%)
SR+ (V/µs)                           92.509       76.275       20.000        ✓ (-17.5%)
CMRR (dB)                            60.949       67.329       60.000        ✓ (+10.5%)
PSRR+ (dB)                           69.974       72.439       60.000        ✓ (+3.5%)
PSRR- (dB)                           66.952       82.260       60.000        ✓ (+22.9%)
IRN (µV)                             22.648       11.139       30.000        ✓ (-50.8%)
Output Swing (V)                      1.609        1.643        1.200        ✓ (+2.1%)
Mismatch 3σ (mV)                     12.512        9.551       15.000        ✓ (-23.7%)

All constraints satisfied: yes

Optimized Transistor OP:
  M1    : Vds=1.1029 Id=60.51µA gm=785.6µS gds=1.71µS saturation
  M2    : Vds=1.0555 Id=60.43µA gm=784.6µS gds=1.76µS saturation
  M3    : Vds=0.4565 Id=60.51µA gm=1207.8µS gds=7.78µS subthreshold     ← weak inversion
  M4    : Vds=0.5040 Id=60.43µA gm=1206.5µS gds=7.40µS subthreshold     ← weak inversion
  M5    : Vds=0.7069 Id=10.00µA gm=125.9µS  gds=0.94µS saturation
  M6    : Vds=0.2405 Id=120.94µA gm=1513.5µS gds=43.75µS saturation
  M7    : Vds=0.8996 Id=142.36µA gm=1814.7µS gds=15.80µS saturation
  M8    : Vds=0.9004 Id=142.36µA gm=1787.4µS gds=11.44µS saturation

Improved over LLM: True
LLM cost: 7.450643, Optimized cost: -1.493860

FINAL_PARAMS = {'M5_L': 1.0, 'M5_WL_ratio': 3.44, 'M5_M': 1, 'M6_M': 13, 'M8_M': 14,
                'M3_L': 1.0, 'M3_WL_ratio': 7.51, 'M3_M': 13,
                'M1_L': 1.0, 'M1_WL_ratio': 9.8,  'M1_M': 10,
                'M7_L': 0.5, 'M7_WL_ratio': 9.68, 'M7_M': 23,
                'C1_value': 2.42e-12, 'Rc_value': 1826.4, 'ibias': 1e-05,
                'M6_L': 1.0, 'M6_WL_ratio': 3.44, 'M8_L': 1.0, 'M8_WL_ratio': 3.44}
TOTAL_TIME = 116.0 s

Excellent! The optimizer found a dramatically better solution. Key change: M3_M: 3 → 13 (diff pair into weak inversion with higher gm). GBW jumped from 50.1 → 73.9 MHz (+48%) while all specs pass.

Note: M3/M4 show "subthreshold" region — this means weak inversion (Vgs < Vth), but the devices ARE saturated (Vds = 0.46 V >> thermal voltage). This is valid and actually beneficial.

Now running extreme PVT on the optimized design.

Bash Extreme PVT simulations on optimized design
OUT
Spec                          Target    ff/20°C    ss/85°C   ff/-40°C
-----------------------------------------------------------------
DC Gain (dB)                    ≥ 70      78.84      78.45      79.24
GBW (MHz)                       ≥ 50      73.94      52.96      95.14
PM (°)                          ≥ 50      56.84      55.51      54.01
Power (µW)                     ≤ 500     491.94     488.23     488.61
SR+ (V/µs)                      ≥ 20      76.27      64.05      93.72
CMRR (dB)                       ≥ 60      67.33      64.69      67.74
PSRR+ (dB)                      ≥ 60      72.44      71.96      73.26
PSRR- (dB)                      ≥ 60      82.26      73.53      81.52
IRN (µV)                        ≤ 30      11.14      12.17      11.04
Swing (V)                      ≥ 1.2       1.64       0.00       1.52
Peaking (dB)                     = 0       0.00       0.00       0.81

OP Flags (optimized design):
  SS/85°C:   all saturated (or weak inversion)
  FF/-40°C:  all saturated (or weak inversion)

6. NUMERICAL OPTIMIZATION

Priority   : GBW (maximize gain-bandwidth product)
Weights    : w_pwr=0.15, w_gain=0.15, w_gbw=1.0
Method     : CMA-ES (λ=16, 6 generations, early stop at 5 stagnant)
Sim calls  : 96 (34 warmup + 96 CMA-ES)
Runtime    : 1.9 min

6a. Parameter Changes

Parameter	LLM sizing	Optimized	Change
M3_M	        3	        13	        +333%

All other parameters unchanged. The optimizer found that increasing the diff pair multiplier M3_M from 3 to 13 dramatically increases gm3 (total W scales 4.3×), pushing the diff pair into weak inversion (gm/ID ≈ 20 S/A) and boosting GBW from 50 to 74 MHz.

6b. Specification Comparison (Optimized)

Spec	        Target	LLM sizing	Optimized	Change	Status
DC Gain (dB)	≥ 70	 77.7	 78.8	 +1.5%	✓
GBW (MHz)	≥ 50	 50.1	 73.9	+47.6%	✓
PM (°)	        ≥ 50	 70.8	 56.8	−19.7%	✓
Power (µW)	≤ 500	483.3	491.9	 +1.8%	✓
SR+ (V/µs)	≥ 20	 92.5	 76.3	−17.5%	✓
CMRR (dB)	≥ 60	 60.9	 67.3	+10.5%	✓
PSRR+ (dB)	≥ 60	 70.0	 72.4	 +3.5%	✓
PSRR- (dB)	≥ 60	 67.0	 82.3	+22.9%	✓
IRN (µV rms)	≤ 30	 22.6	 11.1	−50.8%	✓
Output Swing (V) ≥ 1.2	 1.61	 1.64	 +2.1%	✓
Mismatch 3σ (mV) ≤ 15	 12.5	  9.6	−23.7%	✓

All constraints satisfied: yes

6c. Optimized Sizing Summary

Role          | Device | W(µm)     | L(µm) | M  | ID(µA)  | gm/ID | Vdsat(mV)
LOAD          | M1/M2  | 9.8       | 1.0   | 10 | 60.5    | 13.0  | ~85
DIFF_PAIR     | M3/M4  | 7.51(×L)  | 1.0   | 13 | 60.5    | 20.0  | ~48 (weak inv.)
BIAS_GEN      | M5     | 3.44(×L)  | 1.0   | 1  | 10.0    | 12.6  | ~121
TAIL          | M6     | 3.44(×L)  | 1.0   | 13 | 120.9   | 12.5  | ~121
OUTPUT_CS     | M7     | 9.68(×L)  | 0.5   | 23 | 142.4   | 12.7  | ~91
OUTPUT_BIAS   | M8     | 3.44(×L)  | 1.0   | 14 | 142.4   | 12.6  | ~121

CircuitCollector params:
  M3_L=1.0  M3_WL_ratio=7.51  M3_M=13
  M1_L=1.0  M1_WL_ratio=9.8   M1_M=10
  M5_L=1.0  M5_WL_ratio=3.44  M5_M=1
  M6_M=13   M8_M=14
  M7_L=0.5  M7_WL_ratio=9.68  M7_M=23
  C1_value=2.42pF  Rc_value=1826.4Ω  ibias=10µA

7. EXTREME PVT CHECK (optimized design)

Spec	        Target	ff/20°C	SS/85°C	FF/−40°C
DC Gain (dB)	≥ 70	 78.8	78.5 ✓	79.2 ✓
GBW (MHz)	≥ 50	 73.9	53.0 ✓	95.1 ✓
PM (°)	        ≥ 50	 56.8	55.5 ✓	54.0 ✓
Power (µW)	≤ 500	491.9	488.2 ✓	488.6 ✓
SR+ (V/µs)	≥ 20	 76.3	64.1 ✓	93.7 ✓
CMRR (dB)	≥ 60	 67.3	64.7 ✓	67.7 ✓
PSRR+ (dB)	≥ 60	 72.4	72.0 ✓	73.3 ✓
PSRR- (dB)	≥ 60	 82.3	73.5 ✓	81.5 ✓
IRN (µV)	≤ 30	 11.1	12.2 ✓	11.0 ✓
Swing (V)	≥ 1.2	 1.64	0.00 ❌	1.52 ✓

OP Flags:
  SS/85°C:   All devices saturated (or weak inversion)
  FF/−40°C:  All devices saturated (or weak inversion), 0.81 dB gain peaking

Summary:
  SS/85°C:  9/10 specs pass. Output swing still collapses (same issue as LLM sizing — M6 VDS compression at slow corner).
            However, GBW now passes at SS/85°C (53 MHz ≥ 50 MHz) — a major improvement over the LLM sizing which failed GBW at SS.
  FF/−40°C: 10/10 specs pass ✓

Final Summary
The optimized design delivers 73.9 MHz GBW (48% above target) at the design corner while meeting all 11 active specs. The single key optimization move was increasing the diff pair multiplier M3_M from 3 to 13, which pushes M3/M4 into weak inversion (gm/ID ≈ 20), dramatically boosting transconductance and bandwidth without significant power increase.

The SS/85°C output swing collapse remains an inherent limitation of this topology at extreme slow corners — the M6 tail current source VDS margin is too tight. This would require architectural changes (e.g., LV cascode tail) to resolve.
```

---

## Appendix C: Historical Run Log — `5tota_lv_cascode`

Verbatim Claude Code session sizing `examples/5tota_variants/5tota_lv_cascode.sp` — a 5-transistor OTA with a Sooch-style low-voltage cascode PMOS load (external `Vbias_cas_p` port). Target specs: VDD=1.8 V, CL=5 pF, Gain>40 dB, GBW>50 MHz, PM>60°, Power<500 µW, ff corner @ 20°C, extreme PVT enabled, mismatch and optimization disabled.

This run demonstrates the sub-block LV-cascode abstraction end-to-end: the agent detects the `sub_block_type: 'lv_cascode'` LOAD with `LOAD_CAS` companion role, solves for `Vbias_cas_p`, and tunes it during iteration to gain M5/M6 saturation margin.

```
use analog amplifier skills run 5tota_lv_cascode netlist   use the spec in template
analog-amplifier skill
Read spec-form-template.md

Bash List examples directory
IN
ls /rdf/shared/design_automation/Analog_Sizing/AnalogAgent/examples/
OUT
5tota_variants
tsm_variants

Bash List 5tota variants
IN
ls /rdf/shared/design_automation/Analog_Sizing/AnalogAgent/examples/5tota_variants/
OUT
5tota_cascode.sp
5tota_lv_cascode.sp
5tota_single.sp
README.md

Read 5tota_lv_cascode.sp

Stage [1] Spec Understanding
Read spec-understanding.md

Bash Check LUT temperature availability
IN
python -c "
from scripts.lut_lookup import list_available_L, _resolve_device, _discover_temps
temps = _discover_temps(_resolve_device('nfet'), 'ff')
print('Available NFET FF temps:', temps)
temps = _discover_temps(_resolve_device('pfet'), 'ff')
print('Available PFET FF temps:', temps)
"
OUT
Available NFET FF temps: [-40, 25, 85]
Available PFET FF temps: [-40, 25, 85]

Spec Validation

REQUIRED FIELDS CHECK
======================
VDD  : 1.8 V
CL   : 5e-12 F
Gain : 40 dB
GBW  : 50e6 Hz
PM   : 60°

Status: ALL PRESENT → proceed
LUT temp note: LUT data at 20°C not pre-generated. Using first-order linear interpolation between −40°C and 25°C reference data.


VALIDATED SPECIFICATIONS
========================
Required:
  VDD  = 1.8 V
  CL   = 5e-12 F
  Gain > 40 dB  (100 V/V)
  GBW  > 50e6 Hz
  PM   > 60°

Environment:
  Temperature = 20°C
  Corner      = ff
  VSS         = 0 V

Active Targets (will be optimized):
  Power  < 500e-6 W
  CMRR   > 50 dB
  PSRR+  > 40 dB
  PSRR-  > 50 dB
  I_bias = 10e-6 A

Mismatch:
  Mismatch simulation     : disabled (skipped entirely)

Inactive (blank — will report achieved value only):
  SR+, SR-, IRN, ORN, Output_swing

Post-Sizing:
  Extreme PVT check       : enabled
  Numerical optimization  : disabled

Stage [2] Circuit Understanding
Read circuit-understanding.md

Bash Check existing configs and tools
IN
ls /rdf/shared/design_automation/Analog_Sizing/CircuitCollector/configs/ 2>/dev/null | head -30; echo "---"; ls /rdf/shared/design_automation/Analog_Sizing/AnalogAgent/tools/ 2>/dev/null
OUT
---
api_client.py
bridge_generic.py
bridge.py
bridge_twostage.py
__init__.py
optimizer.py
param_converter.py
__pycache__
topology_manager.py

Bash Find existing topology configs
IN
find /rdf/shared/design_automation/Analog_Sizing/CircuitCollector -name "5tota*" -o -name "netlist.j2" -o -name "testbench.toml" 2>/dev/null | head -30
OUT
…/config/skywater/opamp/5tota_lv_cascode.meta.json
…/config/skywater/opamp/5tota_lv_cascode.toml
…/config/skywater/opamp/5tota_single.toml
…/circuits/opamp/5tota_lv_cascode/netlist.j2
…/circuits/opamp/5tota_single/netlist.j2
  (and various output/5tota_lv_cascode/*.txt files — prior run artifacts)

Read 5tota_lv_cascode.toml
Read netlist.j2

Bash Register 5tota_lv_cascode topology (first attempt, extra_ports=[])
OUT
{'status': 'error', 'message': "extra_ports must be a dict mapping port_name -> DC voltage (e.g. {'Vbias_cas_p': 0.6, 'Vbias_cas_n': 0.6}); got list."}

Bash Register 5tota_lv_cascode topology (fixed: extra_ports={'Vbias_cas_p': 1.0})
OUT
{'status': 'ok', 'config_path': 'config/skywater/opamp/5tota_lv_cascode.toml', 'registered': False}

CIRCUIT IDENTIFICATION
=======================
Topology      : 5T OTA (LV Cascode PMOS Load, Sooch-style)
Match         : circuit-specific/5TOTA/
Devices       : 4 NMOS (M1-M4), 4 PMOS (M5-M8)
Registration  : already_exists (5tota_lv_cascode)
Config path   : config/skywater/opamp/5tota_lv_cascode.toml

Role-Device Map:
  DIFF_PAIR → M1 + [M2]  (nfet)
  TAIL      → M3         (nfet) [mirrors BIAS_GEN]
  BIAS_GEN  → M4         (nfet)
  LOAD      → M5 + [M6]  (pfet)
  LOAD_CAS  → M7 + [M8]  (pfet) [parent LOAD]

Sub-block structures:
  LOAD  : lv_cascode → companion LOAD_CAS → M7
  TAIL  : single
  BIAS_GEN : single

Extra subcircuit ports: Vbias_cas_p
Passive params: none

Activated design flow: circuit-specific/5TOTA/5t-ota-design-flow.md

Stage [3] Circuit-Specific Design Flow
Read 5t-ota-design-flow.md
Read 5t-ota-equation.md

Bash Step 1: DIFF_PAIR sizing
OUT
A0_target = 100.00 V/V
GBW_target = 50.0 MHz

Step 1a: gm1 (initial) = 1.571 mS
Step 1b: (gm/ID)_1 = 12.0 S/A
Available NFET L: [0.18, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
  L=0.18: gm_gds= 17.53  → A0~gm_gds/2 =  8.76
  L=0.5 : gm_gds= 91.71  → A0~gm_gds/2 = 45.86
  L=1.0 : gm_gds=107.00  → A0~gm_gds/2 = 53.50
  L=1.5 : gm_gds=144.88  → A0~gm_gds/2 = 72.44
  L=2.0 : gm_gds=165.83  → A0~gm_gds/2 = 82.91
  L=2.5 : gm_gds=191.69  → A0~gm_gds/2 = 95.85
  L=3.0 : gm_gds=213.94  → A0~gm_gds/2 =106.97
Chosen L1 = 3.0 um
ID1=130.900 uA, I_tail=261.799 uA
W1=111.726 um, L1=3.0 um
gm1=1.571 mS, gds1=7.342 uS, gm/gds=213.94
ft1=0.13 GHz, vdsat1=150.0 mV, vth1=532.1 mV

Bash Steps 2-4: LOAD, cascode, TAIL, analytical check (iter 1, gm/ID_1=12, L1=2.0, L_cas=0.18)
OUT
gm1 = 1.885 mS (20% oversize for parasitics)
ID1=157.08 uA, I_tail=314.16 uA
W1=87.88 um, L1=2.0 um, gm/gds=165.8
P ≈ (I_tail+I_bias)·VDD = 583.5 uW  (spec < 500 uW)

Available PFET L: [0.18, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
  L5=0.18: gm/gds_p=13.69, gds5=137.65uS, A0~103.3
Chosen L5 = 0.18 um
W5=49.69 um, L5=0.18 um
gm5=1.885 mS, gds5=137.649 uS, vdsat5=130.3 mV

LOAD_CAS(M7): W_cas=26.99 um, L_cas=0.18 um, gm_gds_cas=13.09
gds_eq_LOAD=10512.877 nS, p_int_LOAD/2pi=10509.0 MHz
vdsat_cas=167.0mV, vth_cas=762.3mV
p_int/ω_c = 210.2  (need > 3)
Vbias_cas_p = VDD - (vdsat5 + vdsat_cas + |vth_cas|) = 0.740 V

Refined A0 = 86.2 V/V = 38.71 dB  (target 40.00 dB)

M3_M = round(I_tail/I_bias) = round(314.2/10.0) = 31
W3=2.963 um, L3=1.0 um, M3_M=31
gm3=3720.0 uS, gds3=34765.9 nS, vdsat3=141.8 mV
W4=2.963 um, L4=1.0 um, M4_M=1

ANALYTICAL SPEC CHECK (iter 1)
============================================================
A0     :   38.71 dB  (target > 40 dB)   FAIL
GBW    :   58.92 MHz (target > 50 MHz)  OK
PM     :   88.79 deg (target > 60)      OK
fp2    : 1901.00 MHz  (need > 2.2*GBW = 129.6)
p_int  : 10509.01 MHz
SR     :   62.83 V/us
CMRR   :   79.41 dB  (target > 50)      OK
PSRR+  :   38.71 dB  (target > 40)      FAIL
PSRR-  :   79.41 dB  (target > 50)      OK
P      :  583.49 uW  (target < 500)     FAIL
Swing  : 1.215 V,  V_cm_min=0.830 V

Two failures: A0 (38.7 dB) and Power (583 uW). Need to boost gain and cut power.
Fixes: (1) increase L_cas to boost gm_gds_cas → smaller gds_eq_LOAD, (2) bump
gm/ID_1 to 14 to cut tail current (and drop the 20% oversize factor to 10%).

Bash Iter 2: reduce power, boost gain (gm/ID_1=14, L_cas=0.5)
OUT
gm1 = 1.728 mS (10% oversize)
ID1=123.42uA, I_tail=246.84uA, P≈462.3uW (target<500)
W1=101.31um, gm/gds1=176.8

M5: W5=39.04um, L5=0.18um, gm5=1.481mS, gds5=108.153uS, gm/gds=13.69
LOAD_CAS(M7): W_cas=46.46um, L_cas=0.5um, gm/gds_cas=80.72
gds_eq_LOAD=1339.89nS (was 10500), p_int=1641MHz (need>150)
vdsat5=130mV, vdsat_cas=182mV, vth_cas=943mV
Vbias_cas_p = 0.545 V

A0 = 155.5 V/V = 43.83 dB  (target 40)

TAIL(M3): W3=2.963um, M3_M=25, gds3=28037.0nS, vdsat3=142mV

ANALYTICAL SPEC CHECK (iter 2)
============================================================
A0     :   43.83 dB  (>40)  OK
GBW    :   53.81 MHz (>50)  OK
PM     :   87.06 deg (>60)  OK
CMRR   :   84.31 dB  (>50)  OK
PSRR+  :   43.83 dB  (>40)  OK
PSRR-  :   84.31 dB  (>50)  OK
P      :  462.31 uW  (<500) OK
Swing  : 1.223 V,  V_cm_min=0.807 V,  SR=49.4 V/us

All analytical specs pass. Now proceed to Step 5 — Simulation.

Bash Run convert_sizing (first attempt, no re-register in same process)
OUT
convert_sizing result:
{"status": "error", "message": "Unknown topology: '5tota_lv_cascode'. Available: []"}

Bash Inspect bridge_generic topology handling
Bash Find topology dispatch
(Learned that TOPOLOGY_REGISTRY is in-memory per process — must call
 ensure_topology_registered() before convert_sizing in the same python call.)

Bash convert_sizing + simulate_circuit iter 2 (register + convert + simulate in one process)
OUT
register: {'status': 'ok', 'config_path': 'config/skywater/opamp/5tota_lv_cascode.toml', 'registered': False}

convert_sizing:
{
  "status": "ok",
  "params": {
    "M4_L": 1.0,   "M4_WL_ratio": 3.44, "M4_M": 1,
    "M3_L": 1.0,   "M3_WL_ratio": 3.44, "M3_M": 25,
    "M1_L": 2.0,   "M1_WL_ratio": 8.6,  "M1_M": 7,
    "M5_L": 0.18,  "M5_WL_ratio": 9.8,  "M5_M": 26,
    "M7_L": 0.5,   "M7_WL_ratio": 9.6,  "M7_M": 12,
    "ibias": 1e-05
  },
  "config_path": "config/skywater/opamp/5tota_lv_cascode.toml"
}

simulate_circuit (iter 2, Vbias_cas_p=0.545):
Specs:
  dcgain_                = 44.81
  gain_bandwidth_product_= 56.27 MHz
  phase_margin           = 83.60 deg
  cmrr                   = 76.08 dB
  dcpsrp                 = -43.91 dB    (|PSRR+| = 43.91)
  dcpsrn                 = -72.83 dB
  power                  = 431.29 uW
  slew_rate_pos          = 57.08 V/us
  slew_rate_neg          = -36.47 V/us
  vout_low / vout_high   = 0.354 / 1.555 V   → output_swing = 1.202 V
  true_gbw               = 55.96 MHz
  gain_peaking_db        = 0.047

Transistor OP (abbreviated, saturation margins):
  M1: Vds=690 mV, vdsat=117 mV, margin=573 mV        — saturated, large headroom
  M2: Vds=717 mV, vdsat=117 mV, margin=600 mV        — saturated
  M3: Vds=210 mV, vdsat=134 mV, margin=77 mV         — saturated
  M4: Vds=707 mV, vdsat=134 mV, margin=573 mV        — saturated
  M5: Vds=130 mV, vdsat=130 mV, margin=0.4 mV        — edge of saturation ⚠
  M6: Vds=130 mV, vdsat=130 mV, margin=0.2 mV        — edge of saturation ⚠
  M7: Vds=770 mV, vdsat=159 mV, margin=610 mV        — saturated
  M8: Vds=743 mV, vdsat=159 mV, margin=584 mV        — saturated

Stage [4] Simulation & Verification
Checking OP regions: all 8 devices in saturation, but M5/M6 have only
0.2–0.4 mV margin — technically saturated but not robust. One more iteration
to add margin by lowering Vbias_cas_p.

Bash Iter 3: tune Vbias_cas_p for M5/M6 margin
Rationale: V_net_out5 = Vbias_cas_p + |Vgs_M7| ≈ 1.67 V → |Vds_M5|≈130 mV == vdsat5.
Lower Vbias_cas_p by ~80 mV → V_net_out5 drops ~80 mV → |Vds_M5| grows by ~80 mV.
|Vds_M7| drops 770 → 690 mV, still safe (vdsat_cas≈160 mV).

OUT
Vbias_cas_p: 0.545 → 0.465

Specs:
  dcgain_                = 45.65
  gain_bandwidth_product_= 56.32 MHz
  phase_margin           = 83.83 deg
  cmrr                   = 80.51 dB
  dcpsrp                 = -45.22 dB
  dcpsrn                 = -74.42 dB
  power                  = 431.30 uW
  slew_rate_pos          = 62.57 V/us
  slew_rate_neg          = -36.47 V/us
  output_swing           = 1.185 V
  true_gbw               = 56.02 MHz

Device OP (key fields):
  m1: Id=114.83uA, gm/Id=15.47, |Vds|=690mV, vdsat=117mV, sat_margin=573.0mV
  m2: Id=114.78uA, gm/Id=15.48, |Vds|=727mV, vdsat=117mV, sat_margin=609.9mV
  m3: Id=229.61uA, gm/Id=12.43, |Vds|=210mV, vdsat=134mV, sat_margin= 76.7mV
  m4: Id= 10.00uA, gm/Id=12.59, |Vds|=707mV, vdsat=134mV, sat_margin=573.3mV
  m5: Id=114.83uA, gm/Id=11.95, |Vds|=195mV, vdsat=126mV, sat_margin= 68.8mV   ← fixed
  m6: Id=114.78uA, gm/Id=11.95, |Vds|=195mV, vdsat=126mV, sat_margin= 68.5mV   ← fixed
  m7: Id=114.83uA, gm/Id=11.08, |Vds|=705mV, vdsat=160mV, sat_margin=544.8mV
  m8: Id=114.78uA, gm/Id=11.07, |Vds|=668mV, vdsat=160mV, sat_margin=508.2mV

All specs pass with comfortable margins (M5/M6 now at 68 mV sat_margin vs
0.4 mV before). Moving to Extreme PVT.

Stage [5] Extreme PVT Check

Bash Run extreme PVT simulations
OUT

===== FF/20C =====
  A0      = 45.65 dB
  GBW     = 56.32 MHz
  PM      = 83.83 deg
  CMRR    = 80.51 dB
  PSRR+   = 45.22 dB
  PSRR-   = 74.42 dB
  Power   = 431.30 uW
  SR+     = 62.57 V/us
  SR-     = -36.47 V/us
  Swing   = 1.185 V
  OP sat margins (mV):
    m1: 573.0   m2: 609.9   m3:  76.7   m4: 573.3
    m5:  68.8   m6:  68.5   m7: 544.8   m8: 508.2

===== SS/85C =====
  A0      = 45.42 dB
  GBW     = 45.59 MHz         ← below 50 MHz target
  PM      = 84.10 deg
  CMRR    = 71.50 dB
  PSRR+   = 45.23 dB
  PSRR-   = 67.87 dB
  Power   = 421.14 uW
  SR+     = 58.94 V/us
  SR-     = -34.51 V/us
  Swing   = 1.184 V
  OP sat margins (mV):
    m1: 560.9   m2: 603.0   m3:  25.3   m4: 567.8
    m5:  51.7   m6:  51.3   m7: 499.7   m8: 457.9

===== FF/-40C =====
  A0      = 45.23 dB
  GBW     = 63.75 MHz
  PM      = 83.51 deg
  CMRR    = 66.05 dB
  PSRR+   = 42.24 dB
  PSRR-   = 70.87 dB
  Power   = 425.78 uW
  SR+     = 58.58 V/us
  SR-     = -36.19 V/us
  Swing   = 1.160 V
  OP sat margins (mV):
    m1: 622.2   m2: 533.4   m3:  74.7   m4: 624.2
    m5:   7.7   m6:   8.2   m7: 638.6   m8: 726.8

Stage [6] Design Review
Read design-review.md

Bash Step 1 GATE: recompute analytical from final sizes
OUT
Final params: {'M4_L': 1.0, 'M4_WL_ratio': 3.44, 'M4_M': 1,
               'M3_L': 1.0, 'M3_WL_ratio': 3.44, 'M3_M': 25,
               'M1_L': 2.0, 'M1_WL_ratio': 8.6,  'M1_M': 7,
               'M5_L': 0.18,'M5_WL_ratio': 9.8,  'M5_M': 26,
               'M7_L': 0.5, 'M7_WL_ratio': 9.6,  'M7_M': 12,
               'ibias': 1e-05}
Vbias_cas_p: 0.465 V

Analytical (recomputed from FINAL sizes):
{
  "A0_dB":       43.846,
  "GBW_MHz":     63.673,
  "PM_deg":      86.510,
  "SR_Vus":      58.055,
  "CMRR_dB":     84.424,
  "PSRRp_dB":    43.846,
  "PSRRn_dB":    84.424,
  "Power_uW":   540.491,
  "Swing_V":     1.223,
  "V_cm_min":    0.807,
  "M1_W_um":    17.20, "M1_M": 7,  "ID1_uA":  146.67,
  "M5_W_um":     1.76, "M5_M": 26, "ID5_uA":  144.98,
  "M7_W_um":     4.80, "M7_M": 12, "ID_cas":  153.02,
  "M3_W_um":     3.44, "M3_M": 25, "ID3_uA":  290.27,
  "M4_W_um":     3.44, "M4_M": 1,
  "I_tail_uA":  290.27,
  "vdsat1_mV":  123.7, "vdsat5_mV": 130.3,
  "vdsat_cas_mV":181.6,"vdsat3_mV": 141.8
}

==========================================================
DESIGN REVIEW
==========================================================

1. OUTCOME
STATUS: SUCCESS — all active specs met in 3 iterations (analytical sweep + 2 sim iterations)

2. SPECIFICATION COMPLIANCE

Spec     | Target   | Analytical | SPICE      | Error    | Margin   | Status
A0       | > 40 dB  | 43.85 dB   | 45.65 dB   | +4.1%    | +5.65 dB | pass
GBW      | > 50 MHz | 63.67 MHz  | 56.32 MHz  | −11.5%   | +6.32 MHz| pass
PM       | > 60°    | 86.51°     | 83.83°     | −3.1%    | +23.83°  | pass
CMRR     | > 50 dB  | 84.42 dB   | 80.51 dB   | −4.6%    | +30.51 dB| pass
PSRR+    | > 40 dB  | 43.85 dB   | 45.22 dB   | +3.1%    | +5.22 dB | pass
PSRR−    | > 50 dB  | 84.42 dB   | 74.42 dB   | −11.8%   | +24.42 dB| pass
Power    | < 500 µW | 540.5 µW   | 431.3 µW   | −20.2%   | +68.7 µW | pass
I_bias   | = 10 µA  | 10.0 µA    | 10.0 µA    | 0%       | exact    | pass

Reported (no target):
  SR+          = 62.57 V/µs        (analytical: 58.06 V/µs — upper bound; SPICE asymmetric)
  SR−          = −36.47 V/µs       (analytical: 58.06 V/µs — mirror-limited, as expected for 5T OTA)
  Output_swing = 1.185 V           (analytical: 1.223 V)
  Vos (sim)    = 0.19 mV           (~)
  IRN (int., 0.1 Hz – 1 GHz) = 12.5 µV rms  (~)
  ORN (int.)                 = 2.34 mV rms  (~)
  V_cm_min     = 0.373 V           (analytical: 0.807 V — SPICE lower due to body effect)
  V_cm_max     = 1.559 V

Note: analytical I_tail (290 µA) > SPICE (230 µA) because CircuitCollector
quantizes W to the nearest 0.1 µm after WL_ratio rounding. The SPICE result
is ground truth; all specs pass.

3. SIZING SUMMARY

Topology     : 5T OTA with low-voltage (Sooch-style) cascode PMOS load
Process      : SKY130 / ff / 20°C
VDD          : 1.8 V
CL           : 5e-12 F
I_bias       : 10 µA
Vbias_cas_p  : 0.465 V  (external cascode bias port)

Role        | Device | W(µm) | L(µm) | M  | ID(µA) | gm/ID | Vdsat(mV)
DIFF_PAIR   | M1     | 17.2  | 2.0   | 7  | 114.8  | 15.47 | 117
DIFF_PAIR   | M2     | 17.2  | 2.0   | 7  | 114.8  | 15.48 | 117
LOAD        | M5     |  1.76 | 0.18  | 26 | 114.8  | 11.07 | 130
LOAD        | M6     |  1.76 | 0.18  | 26 | 114.8  | 11.07 | 130
LOAD_CAS    | M7     |  4.80 | 0.50  | 12 | 114.8  | 11.08 | 160
LOAD_CAS    | M8     |  4.80 | 0.50  | 12 | 114.8  | 11.07 | 160
TAIL        | M3     |  3.44 | 1.0   | 25 | 229.6  | 12.43 | 134
BIAS_GEN    | M4     |  3.44 | 1.0   | 1  |  10.0  | 12.59 | 134

All 8 devices in saturation (min sat_margin = 68.5 mV on M6).

CircuitCollector params:
  M1_L = 2.0,    M1_WL_ratio = 8.6,   M1_M = 7
  M3_L = 1.0,    M3_WL_ratio = 3.44,  M3_M = 25
  M4_L = 1.0,    M4_WL_ratio = 3.44,  M4_M = 1
  M5_L = 0.18,   M5_WL_ratio = 9.8,   M5_M = 26
  M7_L = 0.5,    M7_WL_ratio = 9.6,   M7_M = 12
  ibias = 10 µA
  Vbias_cas_p = 0.465 V

4. ITERATION HISTORY

Iter | Change Made                                                          | A0(dB) | GBW(MHz) | PM(°) | Power(µW) | Decision
1    | Initial analytical: gm/ID_1=12, L1=2, L5=0.18, L_cas=0.18            | 38.71  | 58.92    | 88.8  | 583       | fail: A0, PSRR+, Power (analytical)
2    | gm/ID_1→14 (cut Itail), L_cas 0.18→0.5 (boost gm_gds_cas)            | 44.81  | 56.27    | 83.6  | 431       | pass specs; M5/M6 sat_margin=0.4 mV (thin)
3    | Vbias_cas_p 0.545 V→0.465 V (add M5/M6 |Vds| margin)                 | 45.65  | 56.32    | 83.8  | 431       | pass — all devices robustly saturated (≥68 mV)

5. EXTREME PVT CHECK

Spec         | Target   | Design (ff/20°C) | SS/85°C    | FF/−40°C
A0           | > 40 dB  | 45.65 dB         | 45.42 dB   | 45.23 dB
GBW          | > 50 MHz | 56.32 MHz        | 45.59 MHz  | 63.75 MHz
PM           | > 60°    | 83.83°           | 84.10°     | 83.51°
CMRR         | > 50 dB  | 80.51 dB         | 71.50 dB   | 66.05 dB
PSRR+        | > 40 dB  | 45.22 dB         | 45.23 dB   | 42.24 dB
PSRR−        | > 50 dB  | 74.42 dB         | 67.87 dB   | 70.87 dB
Power        | < 500 µW | 431.3 µW         | 421.1 µW   | 425.8 µW
SR+          | —        | 62.57 V/µs       | 58.94 V/µs | 58.58 V/µs
SR−          | —        | −36.47 V/µs      | −34.51 V/µs| −36.19 V/µs
Output_swing | —        | 1.185 V          | 1.184 V    | 1.160 V

OP Flags (devices leaving saturation):
  SS/85°C  : all saturated (min margin = 25.3 mV on M3)
  FF/−40°C : all saturated (min margin =  7.7 mV on M5 — thin but positive)

Summary:
  SS/85°C  : 7/8 specs met | GBW drops to 45.6 MHz (−4.4 MHz below 50 MHz target).
             Root cause: reduced µ at high T + reduced drive at SS corner reduce gm1.
             All other specs still pass with margin. Design is not robust for SS/85°C
             if GBW is the hard limit.
  FF/−40°C : 8/8 specs met | GBW improves (+7.4 MHz), M5 saturation margin shrinks to
             8 mV (fast corner increases |Vgs_M7| → pushes M5 drain slightly up toward VDD).
             Positive but thin — consider lowering Vbias_cas_p by another ~10−20 mV for
             cross-corner robustness.

Recommendation: the sizing meets all user-specified specs at the target corner
(ff/20°C). For SS/85°C GBW robustness, ~10−15% higher gm1 (≈125 µA per diff-pair
arm instead of 115) would restore the 50 MHz target across all corners; cost is
~+50 µW of power (still within the 500 µW budget).

Summary
The 5T OTA with low-voltage Sooch-style cascode PMOS load was sized in 3 iterations
against the spec form at spec-form-template.md. All 8 required/active specs pass at
the target corner FF/20°C with healthy margins:
  A0    = 45.65 dB  (target > 40),   GBW = 56.3 MHz (target > 50),   PM = 83.8°
  Power = 431 µW    (target < 500),  CMRR = 80.5 dB,   PSRR+/− = 45.2 / 74.4 dB
  All devices saturated, min margin 68 mV; cascode bias Vbias_cas_p = 0.465 V
PVT check shows GBW drops to 45.6 MHz at SS/85°C (below target) — a known limit of
this sized design; addressed in the report's recommendation.

```